# 语言处理方法的演进：从规则与概率到生成式预训练

「小猫坐在垫子上。」对人来说几乎不需要解释。对机器，这句话却包含几类不同的问题：**哪些字组成一个词？谁在做什么？一个词有歧义时选哪个解释？读到「小猫坐在」时，后面更可能出现什么？**

**自然语言处理（Natural Language Processing，NLP）**的发展，可以从人们如何把这些问题变成可计算的对象来理解。有人编写词典和规则，有人为句子标注结构，有人从语料中统计概率，后来又有人让神经网络学习表示。每一种选择都与当时的数据、算力和应用目标有关，也留下了新的难题。

这次分享沿着「**显式编写知识 → 从出现次数学习 → 学习表示与记忆 → 按需读取上下文 → 通用预训练**」这条思想线展开，最终走向 **GPT（Generative Pre-trained Transformer，生成式预训练变换器）**。其中 **Transformer（变换器）**是一种神经网络架构的名称，后文会展开其计算过程。

这是一条理解方法的路线。真实历史有交叉与并行发展：循环网络和长短期记忆网络的代表工作早于 2003 年的神经语言模型，语义标注也一直延续到深度学习时代。接下来先从历史背景提出问题，再用交互图形观察变化，最后对照公式和代码。

| 阶段 | 核心问题 | 交互实验 |
|---|---|---|
| 1. 规则与标注 | 怎样明确表达语言知识和任务答案？ | 主动句、被动句与语义角色图 |
| 2. 概率语言模型 | 能否从语料估计下一字的概率？ | 上下文长度、计数与平滑 |
| 3. 隐状态序列模型 | 看见词，怎样推断未观测的标签序列？ | 转移概率与整条最优路径 |
| 4. 连续表示 | 未见过的组合，能否借用相近组合的经验？ | 二维向量与下一字概率 |
| 5. 循环网络与记忆门 | 如何保留长度不固定的历史？ | 循环状态、梯度与记忆通路 |
| 6. 翻译中的注意力 | 生成不同译词时，怎样重新读取源句？ | 查询方向、连线与向量加权 |
| 7. Transformer | 如何交换上下文信息，并减少循环依赖？ | 查询、键、值与因果掩码 |
| 8. GPT | 文本自身提供的预测任务，怎样支持迁移学习？ | 模型中间值、采样与参数更新 |
| 9. 方法比较与训练实验 | 各种方法的能力与局限如何判断？ | 上下文可见范围与小模型训练 |

**实验说明**：历史叙述附原始出处；「手工算例」的数值由人工设定；「MiniGPT 实际计算」的数值来自本文实现的小型 GPT 模型。六句练习文本用于展示计算过程，模型在这些文本上的表现不足以说明其具备一般语言理解能力。

<details><summary>运行与演示说明</summary>

本文件是一份 Jupyter Notebook（交互式笔记本），使用本仓库的 Docker 容器环境运行。重启内核后，从头顺序执行即可；使用 CPU（Central Processing Unit，中央处理器），无需下载外部模型或数据。绘图辅助代码默认折叠，核心计算保留在正文。交互控件需要内核保持运行，参考链接需要联网。末尾的完整训练为选做实验。

</details>


In [1]:
import copy
import html
import math
from collections import Counter, OrderedDict
from itertools import product

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

# 这只是本内核的 CPU 设置；小模型通常不需要大量线程。
torch.set_num_threads(min(2, torch.get_num_threads()))
torch.manual_seed(7)
np.set_printoptions(precision=3, suppress=True)
print(f"环境就绪：PyTorch {torch.__version__}；CPU；无外部模型下载")


环境就绪：PyTorch 2.14.1+cpu；CPU；无外部模型下载


In [2]:
# 可折叠工具：只负责显示。后面的数学与模型计算均在正文展开。
PANELS = {}
FLOW = ['字符与 ID', '输入向量 + 位置', 'Transformer Blocks', '词表分数 / 概率', '选择并接回']

def array(value):
    return value.detach().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)

def number(value):
    if isinstance(value, (float, np.floating)):
        return f'{value:.3f}'
    return str(value)

def table(headers, rows, title=None):
    th = ''.join(f'<th style="text-align:left;padding:6px;border-bottom:2px solid #ccd;white-space:nowrap">{html.escape(str(x))}</th>' for x in headers)
    body = ''.join('<tr>'+''.join(f'<td style="padding:6px;border-bottom:1px solid #eef;white-space:nowrap">{html.escape(number(x))}</td>' for x in row)+'</tr>' for row in rows)
    title_html = f'<b>{html.escape(title)}</b>' if title else ''
    display(HTML(f'<div style="overflow:auto;margin:10px 0">{title_html}<table style="font-variant-numeric:tabular-nums;text-align:left">{th}{body}</table></div>'))

def vector_grid(values, labels, title, limit=8):
    a = array(values)
    if a.ndim == 1:
        a = a[None, :]
    visible = a[:, :limit]
    peak = max(float(np.abs(visible).max()), 1e-8)
    rows = []
    for label, row in zip(labels, visible):
        tds = ''
        for x in row:
            alpha = min(.8, abs(float(x))/peak*.65)
            rgb = '47,114,190' if x >= 0 else '219,113,47'
            tds += f'<td style="padding:7px;white-space:nowrap;background:rgba({rgb},{alpha:.3f});color:#111">{x:.3f}</td>'
        rows.append(f'<tr><th style="padding:7px;text-align:left;white-space:nowrap">{html.escape(str(label))}</th>{tds}</tr>')
    display(HTML(f'<div style="overflow:auto;margin:8px 0"><b>{html.escape(title)}</b>（形状 {a.shape}；显示前 {visible.shape[1]} 列）<table style="font-variant-numeric:tabular-nums">'+''.join(rows)+'</table><small>蓝为非负，橙为负；颜色只比较当前表内的数值大小。</small></div>'))

def bars(values, labels, title, top=None):
    a = array(values).reshape(-1)
    order = np.argsort(-a)[:top] if top else np.arange(len(a))
    rows = []
    for i in order:
        p = float(a[i])
        rows.append(f'<div style="display:flex;align-items:center;gap:10px;margin:4px 0;min-width:425px"><span style="width:85px">{html.escape(str(labels[i]))}</span><div style="width:260px;background:#f0f3f7"><div style="width:{max(0,min(100,p*100)):.2f}%;height:16px;background:#3c78b4"></div></div><span>{p:.3f}</span></div>')
    display(HTML(f'<div style="padding:10px;border:1px solid #dde;margin:10px 0;overflow:auto"><b>{html.escape(title)}</b>'+''.join(rows)+'</div>'))

def roadmap(active):
    boxes = []
    for i, label in enumerate(FLOW):
        color = '#dceeff' if i in active else '#f4f5f7'
        boxes.append(f'<span style="display:inline-block;padding:8px;background:{color};border:1px solid #ccd;border-radius:5px;margin:3px">{html.escape(label)}</span>')
    display(HTML('<div style="line-height:2">'+' → '.join(boxes)+'</div>'))

def loss_plot(losses):
    vals = np.asarray(losses, dtype=float)
    lo, hi = float(vals.min()), float(vals.max())
    span = max(hi-lo, .01)
    points = ' '.join(f'{40+i/max(len(vals)-1,1)*590:.2f},{190-(float(y)-lo)/span*150:.2f}' for i,y in enumerate(vals))
    return f'<svg role="img" aria-label="训练批次平均损失曲线" viewBox="0 0 680 235" style="max-width:700px;width:100%;background:#fafcff"><path d="M40 25 V195 H640" fill="none" stroke="#aaa"/><polyline points="{points}" fill="none" stroke="#3c78b4" stroke-width="2"/><text x="45" y="20" font-size="13">训练批次平均损失</text><text x="3" y="45" font-size="11">{hi:.2f}</text><text x="3" y="193" font-size="11">{lo:.2f}</text><text x="40" y="220" font-size="11">1</text><text x="570" y="220" font-size="11">训练步 {len(vals)}</text></svg>'

def panel(key, render, controls):
    out = widgets.Output()
    state = {'controls': controls, 'renders': 0, 'error': None}
    def refresh(change=None):
        state['renders'] += 1
        state['error'] = None
        with out:
            clear_output(wait=True)
            try:
                render(**{k: v.value for k,v in controls.items()})
            except ValueError as exc:
                state['error'] = str(exc)
                display(HTML(f'<p style="color:#b34">{html.escape(str(exc))}</p>'))
    for control in controls.values():
        control.observe(refresh, names='value')
    state['refresh'] = refresh
    PANELS[key] = state
    refresh()
    ui = widgets.VBox([widgets.VBox(list(controls.values())), out])
    display(ui)
    return ui


In [3]:
# 只绘图；概率、递推和梯度仍在各节正文的代码中计算。
BLUE, ORANGE, GREEN, RED = '#2474ab', '#c35e24', '#218568', '#b23c53'

def svg_text(x, y, text, color='#25313a', size=14, anchor='middle'):
    return (f'<text x="{x:.2f}" y="{y:.2f}" fill="{color}" font-size="{size}" '
            f'text-anchor="{anchor}" font-family="sans-serif">{html.escape(str(text))}</text>')

def svg_arrow(x1, y1, x2, y2, color=BLUE, width=2, opacity=1):
    dx, dy = x2-x1, y2-y1
    norm = max(math.hypot(dx, dy), 1e-8)
    ux, uy = dx/norm, dy/norm
    bx, by = x2-9*ux, y2-9*uy
    return (f'<g opacity="{opacity}"><line x1="{x1}" y1="{y1}" x2="{bx}" y2="{by}" '
            f'stroke="{color}" stroke-width="{width}"/>'
            f'<polygon points="{x2},{y2} {bx-4*uy},{by+4*ux} {bx+4*uy},{by-4*ux}" fill="{color}"/></g>')

def show_svg(parts, width=900, height=330, label='数学关系图'):
    content = ''.join(parts) if isinstance(parts, list) else parts
    display(HTML(f'<div style="overflow:auto;margin:12px 0"><svg role="img" '
                 f'aria-label="{html.escape(label, quote=True)}" viewBox="0 0 {width} {height}" '
                 f'style="width:100%;min-width:{min(width,680)}px;max-width:{min(width,1000)}px;background:#fcfdfd">'
                 f'<title>{html.escape(label)}</title>{content}</svg></div>'))

def vector_plane(vectors, labels, colors, left=0, title='', extent=2., label_offsets=None):
    # 所有平面固定为等比例坐标，向量端点和刻度共用同一变换。
    cx, cy, scale = left+215, 182, 64/extent*2
    parts = [svg_text(left+215, 25, title, size=16)]
    for tick in [-2, -1, 0, 1, 2]:
        p = tick*scale
        parts.append(f'<path d="M{cx+p} 53 V{cy+130} M{left+67} {cy-p} H{left+363}" stroke="#e5e9ed"/>')
        if tick:
            parts.extend([svg_text(cx+p, cy+17, tick, size=11), svg_text(cx-10, cy-p+4, tick, size=11, anchor='end')])
    parts += [svg_arrow(left+55,cy,left+379,cy,'#82909a',1),
              svg_arrow(cx,cy+135,cx,46,'#82909a',1),
              svg_text(left+380,cy-10,'坐标 1',size=12,anchor='end'),
              svg_text(cx+10,50,'坐标 2',size=12,anchor='start')]
    for i,(v,label,color) in enumerate(zip(vectors,labels,colors)):
        x,y = cx+float(v[0])*scale, cy-float(v[1])*scale
        dx,dy = label_offsets[i] if label_offsets else (10,-10-14*(i%2))
        parts.extend([svg_arrow(cx,cy,x,y,color,2.5),svg_text(x+dx,y+dy,label,color,size=13,anchor='start')])
    return parts

def weighted_geometry(query, keys, values, weights, labels):
    parts = vector_plane([query,*keys],['q',*[f'k{i}' for i in range(len(keys))]],
                         [RED,*([BLUE]*len(keys))],title='匹配：点积随方向与长度变化',extent=2.5)
    left, cx, cy, scale = 440, 655, 182, 64
    parts += vector_plane([],[],[],left=left,title='取回：按权重首尾相接',extent=2.)
    cursor = np.zeros(2)
    palette = [BLUE,ORANGE,GREEN,RED]
    for j,(weight,value) in enumerate(zip(weights,values)):
        end = cursor+weight*value
        x1,y1 = cx+cursor[0]*scale,cy-cursor[1]*scale
        x2,y2 = cx+end[0]*scale,cy-end[1]*scale
        parts += [svg_arrow(x1,y1,x2,y2,palette[j%4],3),
                  svg_text(left+22,282+j*18,f'{labels[j]}：{weight:.3f} × v{j}',palette[j%4],12,anchor='start')]
        cursor = end
    parts += [svg_arrow(cx,cy,cx+cursor[0]*scale,cy-cursor[1]*scale,'#161d25',1.5),
              svg_text(left+410,310,f'z = ({cursor[0]:.3f}, {cursor[1]:.3f})',size=13,anchor='end')]
    show_svg(parts,900,370,'查询与键的二维方向，以及加权值向量的首尾相加')


## 历史脉络：把年代与问题放在同一张图上

下方时间轴列出一些代表事件。选择节点，可以查看它所回应的问题；人物与原始工作放到后文对应章节展开。事件按时间排列，节点间距不表示相同年数。


In [4]:
HISTORY = [
    ('1948','统计结构','香农用不同阶统计近似生成符号序列，展示语言中的统计结构。'),
    ('1971','显式世界','积木世界语言理解系统：把语言接到限定世界中的对象和动作。'),
    ('1989','隐状态模型','概率状态模型用于语音识别；1989 年是代表教程的发表年份。'),
    ('1990','循环状态','简单循环网络让上一步状态参与下一步计算。'),
    ('1993','标注语料','宾夕法尼亚树库提供词性与句法树标注，明确结构分析的监督目标。'),
    ('1997','记忆门','长短期记忆网络为跨越长时间间隔的学习设计记忆机制。'),
    ('2003','神经语言模型','联合学习词表示与下一词概率，通过连续表示共享参数。'),
    ('2005','语义角色','命题库在语料中标出谓词与论元，为语义角色分析提供数据。'),
    ('2014','按需读取','神经翻译对源句进行软对齐，按需读取不同位置；会议发表为 2015 年。'),
    ('2017','Transformer','以注意力与逐位置计算构成主干，原版用于编码器—解码器翻译。'),
    ('2018.06','GPT','生成式 Transformer 预训练，再监督微调到多个语言任务。'),
    ('2018.10','双向预训练','双向上下文与遮盖预测：与生成式预训练并行的另一条路线。'),
]

def render_history(event):
    parts = []
    for row in range(2):
        y = 54+row*115
        parts.append(f'<path d="M55 {y} H845" stroke="#9caeb9"/>')
        for col in range(6):
            i, x = row*6+col, 65+col*153
            color = RED if i == event else BLUE
            parts += [f'<circle cx="{x}" cy="{y}" r="{8 if i==event else 5}" fill="{color}"/>',
                      svg_text(x,y-20,HISTORY[i][0],color,13),
                      svg_text(x,y+30,HISTORY[i][1],color,13)]
    show_svg(parts,900,230,'自然语言处理代表事件时间轴，事件间距不代表年数')
    display(HTML(f'<p><b>{html.escape(HISTORY[event][0]+" · "+HISTORY[event][1])}</b>：{html.escape(HISTORY[event][2])}</p>'))


In [5]:
history_ui = panel('history',render_history,OrderedDict(
    event=widgets.Dropdown(options=[(f'{y} · {name}',i) for i,(y,name,_) in enumerate(HISTORY)],
                           value=0,description='历史节点',layout=widgets.Layout(width='min(100%, 520px)'))))

### 两条贯穿线：分析句子结构，与预测后续文字

我们会反复回到「小猫坐在垫子上。」：**结构分析**给它词性或“谁做什么”的标签；**语言建模**在读到「小猫坐在」时，为下一个字符分配概率。两个任务互有关联，答案的形式却不同，也分别只覆盖语言理解的一部分。

下面准备六句小型练习语料，供计数模型、循环网络算例和 **MiniGPT（本文实现的迷你 GPT）**使用。除词性示例按词切分外，预测实验都按**单个字符**处理。

模型处理的基本单位称为**词元（token）**，本例中一个字符就是一个词元。每个词元对应一个 **ID（Identifier，标识符）**，用于查表；编号大小本身不表示语义关系。


In [6]:
SENTENCES = [
    '小猫坐在垫子上。', '小狗坐在门口。',
    '小猫躺在垫子上。', '小狗趴在门口。',
    '小鸟飞在天空中。', '小鱼游在池水里。',
]
CORPUS = ''.join(SENTENCES) * 40
CHARS = list(dict.fromkeys(CORPUS))
STOI = {ch: i for i, ch in enumerate(CHARS)}
ITOS = dict(enumerate(CHARS))
VOCAB_SIZE = len(CHARS)
PREFIX = '小猫坐在'

def encode(text):
    if not text:
        raise ValueError('请输入至少一个字符。')
    unknown = sorted(set(text) - set(STOI))
    if unknown:
        raise ValueError('字符不在本练习词表中：'+ '、'.join(repr(ch) for ch in unknown))
    return torch.tensor([[STOI[ch] for ch in text]], dtype=torch.long)

def decode(ids):
    return ''.join(ITOS[int(i)] for i in ids)

prefix_ids = encode(PREFIX)
table(['位置 t（从 0 开始）', '字符 / token', '词表 ID'],
      [(t, ch, STOI[ch]) for t,ch in enumerate(PREFIX)], '同一个贯穿前缀')
print('练习词表：', ''.join(CHARS), '；大小 V =', VOCAB_SIZE)
assert decode(prefix_ids[0]) == PREFIX


0,小,0
1,猫,1
2,坐,2
3,在,3


练习词表： 小猫坐在垫子上。狗门口躺趴鸟飞天空中鱼游池水里 ；大小 V = 23


## 1 · 规则、词典与标注：先明确任务与语言结构

**位置**：从人工设计的显式表示开始。**问题**：如何把字符串接到对象、动作与关系上？**输入 → 输出**：词和句式 → 显式结构。**与下一节的联系**：规则覆盖不足，歧义该怎么处理？

设想一个只能操作积木的房间。物体、颜色、位置关系和动作都很有限；如果把词汇、语法、指代与这个小世界的状态接起来，机器便能执行部分自然语言命令。

**特里·维诺格拉德（Terry Winograd）**是自然语言理解与人机交互研究者、斯坦福大学计算机科学荣休教授。他早期在麻省理工学院开发了 **SHRDLU（积木世界语言理解系统）**，把语言分析、推理与动作执行连接起来。SHRDLU 是系统专名，并非需要展开的英文首字母缩写。[人物背景](https://profiles.stanford.edu/terry-winograd)

维诺格拉德 1971 年学位论文中的系统有一个重要前提：任务世界经过仔细限定。开放语言中，新词、隐喻、背景知识和改写方式会不断越过边界。[原始论文与项目说明](https://dspace.mit.edu/entities/publication/3406b826-5a41-4b72-9430-7876c0d5405e)

于是我们需要先说清“希望机器预测哪种结构”。**词性标注**回答某个词在句中属于什么类别；**句法分析**描述组合和依存关系；**语义角色标注**描述围绕谓词的论元，例如谁实施动作、谁承受动作。标注既可能由人提供监督答案，也可能由模型预测。它是一类任务和数据资源，不能看成在概率序列模型出现后便结束的一代算法。

**宾夕法尼亚树库（Penn Treebank）**和**命题库（Proposition Bank，PropBank）**是两个相关的标注语料库/项目，不是人物。前者的 1993 年论文介绍词性与句法标注；后者的 2005 年论文在相关语料基础上补充谓词—论元层面的语义角色标注。这个层次也不是完整的世界知识或句意。[Penn Treebank](https://aclanthology.org/J93-2004/)、[PropBank](https://aclanthology.org/J05-1004/)

**先预测**：把「小猫追小狗」改成「小狗被小猫追」，第一个名词还是动作发出者吗？下面的简化规则把“第一个名词”当作施事；切换句子，再切换词性与角色层，观察机器输出与人工答案的区别。此处用简化案例呈现显式角色表示，未复现 SHRDLU 系统。


In [7]:
RULE_CASES = [
    dict(text='小猫坐在垫子上。',words=['小猫','坐','在','垫子','上','。'],
         pos=['名词','动词','介词','名词','方位词','标点'],agent='小猫',predicate='坐',role='地点',argument='垫子上'),
    dict(text='小猫追小狗。',words=['小猫','追','小狗','。'],pos=['名词','动词','名词','标点'],
         agent='小猫',predicate='追',role='受事',argument='小狗'),
    dict(text='小狗被小猫追。',words=['小狗','被','小猫','追','。'],pos=['名词','介词','名词','动词','标点'],
         agent='小猫',predicate='追',role='受事',argument='小狗'),
]


def rule_agent(item):
    return next(word for word,pos in zip(item['words'],item['pos']) if pos=='名词')

In [8]:
def render_rules(case, level):
    item = RULE_CASES[case]
    table(['词','人工给定的词性'],list(zip(item['words'],item['pos'])),'固定的小型人工词性表；没有训练标注器')
    if level == '词性':
        print('词性说这个词在本句中的类别，尚未说明谁追谁。')
        return
    predicted = rule_agent(item)
    role_rows = [('施事',item['agent']),('谓词',item['predicate']),(item['role'],item['argument'])]
    parts = []
    for i,(role,word) in enumerate(role_rows):
        x=150+i*290
        parts += [f'<rect x="{x-100}" y="90" width="200" height="75" rx="5" fill="#edf4f8" stroke="#c1d1db"/>',
                  svg_text(x,65,role,[BLUE,ORANGE,GREEN][i],15),svg_text(x,135,word,size=22)]
    parts += [svg_arrow(260,127,330,127,BLUE),svg_arrow(550,127,620,127,GREEN)]
    show_svg(parts,900,205,'人工给定的谓词与角色关系')
    table(['输出来源','施事','是否与人工施事一致'],[
        ['简化规则：第一个名词',predicted,predicted==item['agent']],
        ['人工参考答案',item['agent'],True]])
    print('这里图中的角色来自人工答案。简化规则的失败告诉我们需要改模型或规则，不会自动产生新知识。')

In [9]:
rules_ui = panel('rules',render_rules,OrderedDict(
    case=widgets.Dropdown(options=[(x['text'],i) for i,x in enumerate(RULE_CASES)],value=0,description='句子'),
    level=widgets.ToggleButtons(options=['词性','语义角色'],value='语义角色',description='观察层')))

<details><summary>解释与下一步</summary>
被动句中，名词先后顺序改变了，动作参与者未必改变。我们当然可以增加“被字句”规则，但语言不断产生变体。另一条思路是：让候选解释竞争，再从数据估计哪个更可能。这需要概率；在进入隐藏标签之前，我们先从最容易统计的下一字任务开始。
</details>


## 2 · 语言也是概率：从猜下一字到数据稀疏

**位置**：把任务变成对候选结果分配概率。**问题**：不手写所有续写规则，能不能从出现次数学习？**输入 → 输出**：末尾一小段上下文 → 词表分布。**与下一节的联系**：该看的结构看不见，长组合又很少重复。

**克劳德·埃尔伍德·香农（Claude Elwood Shannon，1916—2001）**是美国数学家、工程师，曾在贝尔实验室和麻省理工学院工作，是信息论的奠基者。他关心的一个基本问题是：信息中的不确定性怎样度量，又怎样可靠传递？[人物背景](https://news.mit.edu/2001/shannon)

1948 年，香农让字母和词按不同阶的统计关系产生序列：上下文越丰富，生成的文本越接近自然语言。这说明**语言中存在可利用的统计结构**，也把语言预测与信息度量联系起来：字并不是独立出现的；看过前面的字，会改变后面各字的不确定性。[原始论文](https://people.math.harvard.edu/~ctm/home/text/others/shannon/entropy/entropy.pdf)

在语言建模中，整句概率可以沿条件概率链拆开：

$$P(x_1,\ldots,x_T)=\prod_{t=1}^T P(x_t\mid x_1,\ldots,x_{t-1}).$$

这只是概率的链式法则，没有规定必须使用哪种模型。难处在每一项如何估计。**n 元语言模型（n-gram language model，简称 N-gram）**只保留最近的 $k=n-1$ 个词元；其中 n 表示一个统计片段包含的词元数：

$$P(x_t\mid x_{<t})\approx P(x_t\mid x_{t-k:t}),\qquad
\hat P(w\mid c)=\frac{C(c,w)+\alpha}{C(c)+\alpha |V|}.$$

这里 $C(c,w)$ 是上下文 $c$ 后出现 $w$ 的次数，$C(c)$ 是这些次数的总和，$|V|$ 是候选字符数。$\alpha=0$ 是直接数频率；正数是最简单的加法平滑。平滑避免零概率，但没有凭空读懂未见组合的意思。

**先预测**：「小猫坐在」与「小狗坐在」的末尾两个字相同。只看最近两字，能把两句的续写分开吗？把窗口改成四字，再换成语料从未出现的「小猫趴在」，又会怎样？


In [10]:
def ngram_distribution(prefix, context_length, alpha):
    context = prefix[-context_length:] if context_length else ''
    counts = Counter()
    for sentence in SENTENCES:  # 不跨越句界统计上下文。
        for t in range(context_length,len(sentence)):
            if sentence[t-context_length:t] == context:
                counts[sentence[t]] += 1
    count_vector = np.array([counts[ch] for ch in CHARS],dtype=float)
    denominator = count_vector.sum()+alpha*VOCAB_SIZE
    probabilities = (count_vector+alpha)/denominator if denominator else None
    return context,count_vector,probabilities

In [11]:
def render_ngram(prefix, context_length, alpha):
    context,counts,probabilities = ngram_distribution(prefix,context_length,alpha)
    print(f'只读取上下文 {context!r}；匹配到 {int(counts.sum())} 个后继；完整前缀 {prefix!r}')
    table(['候选','出现次数 C(c,w)','平滑后分子'],[
        [ch,int(counts[i]),float(counts[i]+alpha)] for i,ch in enumerate(CHARS) if counts[i]>0],
        '表中只列实际出现过的后继；平滑作用于整个 23 字词表')
    if probabilities is None:
        print('没有观测且未平滑，分母为 0，经验分布未定义。提高 α 才能得到本例的平滑分布。')
    else:
        bars(probabilities,CHARS,'计数估计的下一字符概率',top=8)
        assert np.isclose(probabilities.sum(),1.)

In [12]:
ngram_ui = panel('ngram',render_ngram,OrderedDict(
    prefix=widgets.Dropdown(options=['小猫坐在','小狗坐在','小猫趴在'],value=PREFIX,description='前缀'),
    context_length=widgets.IntSlider(value=2,min=0,max=4,description='看前 k 字',continuous_update=False),
    alpha=widgets.FloatSlider(value=0,min=0,max=1,step=.1,description='平滑 α',continuous_update=False)))

<details><summary>解释与下一步</summary>
同样的后缀「坐在」得到相同的计数分布，无法区分更早出现的猫和狗。长窗口能区分训练中见过的组合，却更容易遇到零计数。接下来有两条路线：若任务是为词标注类别，可把类别当成隐藏变量，用概率序列模型推断；若任务是下一词预测，可让相近词共享参数，走向连续表示。它们解决的不是同一个问题，因而不应直接比较两个任务的概率数值。
</details>


## 3 · 隐马尔可夫模型：从观测词推断隐藏词性

**当前位置**：从人工规则走向概率序列模型。**待解决的问题**：一个词有多个词性时，怎样同时利用这个词本身的线索和相邻词的搭配？**输入**是词序列，**输出**是最可能的整条词性路径；这与“预测下一个字符”是不同任务。

**隐马尔可夫模型（Hidden Markov Model，HMM）**可以先理解为一台“看不见内部标签的打分器”：我们看见词（观测），看不见词性（隐藏状态），模型用概率给每条可能的词性路径打分。

其中“马尔可夫”来自**安德烈·安德烈耶维奇·马尔可夫（Andrei Andreyevich Markov，1856—1922）**。他是研究概率论的俄国数学家，以描述相依随机变量序列的马尔可夫链闻名。隐马尔可夫模型继承了“下一状态主要由当前状态决定”的假设，并加入观测变量。[人物背景](https://mathshistory.st-andrews.ac.uk/Biographies/Markov/)

下面先单独把**马尔可夫链**说清楚，再进入带隐藏标签的 HMM。

### 3.1 马尔可夫链：下一状态只看当前

**马尔可夫链（Markov chain）**是一条随时间推进的随机变量序列 $X_1,X_2,\ldots$，核心思想只有一句：

> **未来与过去无关，只通过“现在”发生联系。**

写成公式，就是一阶马尔可夫假设：

$$P(X_t\mid X_{1:t-1})=P(X_t\mid X_{t-1}).$$

没有这条假设时，链式法则仍然成立，但条件里要带着全部历史 $X_{1:t-1}$，需要估计的完整历史条件数量会随序列长度快速增长。马尔可夫思想刻意把依赖压短：下一步只查一张**转移表** $A_{ij}=P(X_t=j\mid X_{t-1}=i)$，再配上初始分布 $\pi_i=P(X_1=i)$，整条序列的联合概率就变成

$$P(X_{1:T})=\pi_{X_1}\prod_{t=2}^{T}A_{X_{t-1},X_t}.$$

**为什么愿意这样简化？** 不是因为真实世界真的“只有一步记忆”，而是用可计算的局部依赖，去近似仍然相依的序列。天气、棋盘格子、网页跳转都可以先建成可见状态的马尔可夫链：你看到的就是 $X_t$ 本身。

和第 2 站的联系也很直接：**bigram 语言模型就是词上的一阶马尔可夫链**；n-gram 则是把“当前状态”扩成最近 $n-1$ 个词，成为更高阶的马尔可夫模型。它们估的是“下一词是什么”。

HMM 并没有丢掉这条思想，而是把它用在**看不见的标签序列** $z_t$ 上，再让每个 $z_t$ 按发射概率产生观测词 $w_t$。可以记：

$$\text{HMM}=\text{隐状态上的马尔可夫链}+\text{观测发射}.$$

因此后文的转移矩阵 $A$ 仍是马尔可夫链那一套；“隐”指状态 $z_t$ 没有被直接观测；发射矩阵 $E$ 负责把这个隐藏状态与观测词 $w_t$ 连接起来。

### 3.2 先把一句歧义话拆成五个量

`Time flies` 可以读成“时间飞逝”（名词 N → 动词 V），也可以在特殊语境中读成“给苍蝇计时”（动词 V → 名词 N）。这里：

| 符号 | 在本例中代表什么 | 直观问题 |
|---|---|---|
| $w_t$ | 第 $t$ 个观测词，如 `time`、`flies` | 我实际看到了什么？ |
| $z_t$ | 第 $t$ 个隐藏词性，如 N、V | 这个词在句中扮演什么角色？ |
| $\pi_j$ | 第一个词的词性是状态 $j$ 的概率 | 一开头更常见哪种词性？ |
| $A_{ij}$ | 词性 $i$ 后面接词性 $j$ 的概率 | 前一个词性会怎样影响下一个？ |
| $E_{j,w}$ | 词性 $j$ 产生词 $w$ 的概率 | 这种词性通常会出现哪些词？ |

$w_t$ 是输入，$z_t$ 是要求解的隐藏标签；但打分还需要 $(\pi,A,E)$。它们不是维特比推出来的，而是**事先从数据估计（训练）**，再拿去给路径打分。

**有人工词性时（有监督）**：在标注语料上数频率，再按行归一化。记 $\#$ 为计数：

$$\hat\pi_j=\frac{\#(\text{句首为 }j)}{\#(\text{句子})},\quad
\hat A_{ij}=\frac{\#(i\to j)}{\#(i\to\cdot)},\quad
\hat E_{j,w}=\frac{\#(z=j,\,w)}{\#(z=j)}.$$

未见过的词或转移通常再加平滑，避免概率为 0。这把第 1 站的标注答案接到了模型参数上。

**只有词、没有词性时（无监督）**：无法直接数上述次数，常用期望最大化（HMM 里称 Baum–Welch）：先初始化一套 $(\pi,A,E)$ → 用当前参数给每一步词性打软计数（后验）→ 用软计数按同一套频率公式更新参数 → 迭代。这样能抬高“生成观测”的似然，但不保证学到的隐状态自动等于语言学上的 N/V。

本算例后文的数值是**手工设定的演示参数**，未执行上述训练；目的是先把路径分数与解码算清楚。

**劳伦斯·拉宾纳（Lawrence Rabiner）**长期研究数字信号与语音处理，曾在贝尔实验室从事语音研究；他在 **1989 年**发表的教程系统介绍了 HMM 的方法与语音识别应用，**1989 年不是 HMM 的发明时间**。[人物背景](https://web.ece.ucsb.edu/Faculty/Rabiner/)、[教程原文](https://www.cs.cornell.edu/courses/cs481/2004fa/rabiner.pdf)

**朱利安·库皮克（Julian Kupiec，中文为音译）**是从事词性标注、双语语料等研究的自然语言处理研究者，曾在施乐帕洛阿尔托研究中心工作；其 **1992 年**论文展示了 HMM 词性标注。[相关研究工作](https://aclanthology.org/P93-1003/)、[词性标注论文](https://doi.org/10.1016/0885-2308(92)90019-Z)

这些工作处理结构化序列中的不确定性；本节关注其中的词性标注问题。

### 3.3 一条路径怎样得到分数？

设隐藏状态为 $z_t$，观测词为 $w_t$。一条路径的联合概率是：

$$P(z_{1:T},w_{1:T})=\underbrace{\pi_{z_1}}_{先选第一个词性}\underbrace{E_{z_1,w_1}}_{第一个词由它产生}\prod_{t=2}^{T}\underbrace{A_{z_{t-1},z_t}}_{词性转移}\underbrace{E_{z_t,w_t}}_{当前词的发射}.$$ 

模型做了两个简化：当前状态只依赖前一个状态（正是 3.1 的马尔可夫思想，用在隐藏词性 $z_t$ 上）；已知当前状态后，当前词不再额外依赖其他状态和词。简化让计算可行，也限制了模型能表达的语言关系。

先给出本算例的参数——它们是**手工设定的词性歧义算例，未经过语料训练**。发射矩阵的列为 `time, flies, other`，其中 `other` 表示其他词；两行分别为 N、V：

$$E=\begin{bmatrix}.45&.45&.10\\.45&.40&.15\end{bmatrix},\quad
\pi=[s,1-s],\quad A=\begin{bmatrix}1-a&a\\.9&.1\end{bmatrix}.$$

默认取 $s=.6,\,a=.5$，于是

$$\pi=[.6,.4],\qquad A=\begin{bmatrix}.5&.5\\.9&.1\end{bmatrix}.$$

本算例只看两步。路径 `V → N` 对应四个因子，可直接从表中读出：

| 因子 | 查表 | 数值 |
|---|---|---|
| $\pi_V$ | $\pi$ 的第二项 $1-s$ | $.4$ |
| $E_{V,\text{time}}$ | $E$ 第 2 行第 1 列 | $.45$ |
| $A_{V\to N}$ | $A$ 第 2 行第 1 列 | $.9$ |
| $E_{N,\text{flies}}$ | $E$ 第 1 行第 2 列 | $.45$ |

$$P(VN,\text{time flies})=.4\times.45\times.9\times.45=.0729.$$

这个乘法把四个问题依次相乘：第一词性是什么、它能否产生 `time`、下一个词性怎么转移、它能否产生 `flies`。

逐步贪心先选 N：$.6\times.45=.27>.4\times.45=.18$；再从 N 选 N：$.5\times.45>.5\times.40$，得到 NN。完整路径 NN 的联合概率是 $.6\times.45\times.5\times.45=.06075$，低于 VN 的 `.0729`。**局部一步最高，不保证整条路径最高。**

### 3.4 维特比：只保留“到这里为止最好”的路径

**维特比算法（Viterbi algorithm）**由通信工程师**安德鲁·维特比（Andrew Viterbi）**提出。它不保留所有完整路径，而是对每一个“时间位置 × 当前词性”只保留一条最好前缀，并记下这条前缀来自哪个上一步状态。[人物背景](https://viterbischool.usc.edu/about-andrew-viterbi/)

记 $\delta_t(j)$ 为“读完前 $t$ 个词，并且第 $t$ 个词性是 $j$ 时，所有前缀中的最高对数分数”。因此递推式可以逐项读成：

$$\delta_1(j)=\log\pi_j+\log E_{j,w_1},$$
$$\delta_t(j)=\log E_{j,w_t}+\max_i\{\delta_{t-1}(i)+\log A_{ij}\}.$$

先从每个上一步状态 $i$ 走到当前状态 $j$，挑分数最高的那个；再加上当前词的发射分数。`backpointers` 保存“挑中了谁”，最后从最高终点反向走回去。

**为什么这个递推成立？** 它是动态规划：状态是“时刻 $t$、当前词性 $j$”，目标是到该格的最高对数联合分数。任意以 $z_t=j$ 结尾的路径，分数都可拆成三段：

$$\underbrace{\text{停在 }z_{t-1}=i\text{ 的前缀分数}}_{\delta_{t-1}(i)\text{ 只保留其中最大者}}+\log A_{ij}+\log E_{j,w_t}.$$

最后一项对所有“终点为 $j$”的路径相同，可提到 $\max$ 外。中间的转移 $A_{ij}$ 只依赖上一词性 $i$（3.1 的马尔可夫假设），不依赖更早的 $z_1,\ldots,z_{t-2}$。因此：若有一条前缀停在同一 $i$ 却分数更高，换成它只会让到达 $j$ 的整条路径更好——这就是**最优子结构**。于是每个 $i$ 只需保留 $\delta_{t-1}(i)$，再对 $i$ 取 $\max$，得到上面的递推；更差前缀可以安全丢掉。若下一状态还依赖更早历史，按单个 $i$ 分桶就不够，递推式必须改写。

为什么要取对数？原始路径要连续相乘，数字很快变得极小；对数把乘法变成加法，最大路径不变，计算更稳定。下面的 `delta` 是最佳前缀的对数联合分数，不是当前词性的后验概率；四条路径的联合概率之和也不要求等于 1。

**先预测 → 操作 → 解释**：先猜把 `N → V` 从 `.5` 调到 `.8` 会改变哪几条路径；再移动滑块，看绿色路径与分数表；最后说明为什么修改转移概率不会修改发射矩阵。


In [13]:
# 核心计算保留在正文：状态顺序 N、V；词表顺序 time、flies、other。
def viterbi(observations, initial, transition, emission):
    """用动态规划寻找观测序列的最高分状态路径。"""
    observations = np.asarray(observations, dtype=int)
    initial = np.asarray(initial, dtype=float)
    transition = np.asarray(transition, dtype=float)
    emission = np.asarray(emission, dtype=float)
    assert len(observations) > 0 and np.isclose(initial.sum(), 1)
    assert np.allclose(transition.sum(axis=1), 1)
    assert np.allclose(emission.sum(axis=1), 1)
    # 概率连乘容易下溢；取 log 后，乘法变成加法。
    with np.errstate(divide="ignore"):
        log_pi, log_a, log_e = np.log(initial), np.log(transition), np.log(emission)
    # delta[t, j]：读到第 t 个词、当前状态为 j 时的最佳前缀分数。
    delta = np.full((len(observations), len(initial)), -np.inf)
    # 回溯时需要知道：到达当前状态 j 的最佳前一步状态是谁。
    backpointers = np.full(delta.shape, -1, dtype=int)
    # 第一个词没有“转移”，只有初始概率 × 发射概率。
    delta[0] = log_pi + log_e[:, observations[0]]
    for t in range(1, len(observations)):
        # candidates[i, j] 是“上一步 i → 当前 j”的所有候选前缀分数。
        candidates = delta[t - 1, :, None] + log_a
        # 每一列只保留到达当前状态 j 的最佳上一步。
        backpointers[t] = np.argmax(candidates, axis=0)
        delta[t] = candidates[backpointers[t], np.arange(len(initial))] + log_e[:, observations[t]]
    # 终点取最高分，然后沿回指针倒着恢复整条路径。
    path = [int(np.argmax(delta[-1]))]
    for t in range(len(observations) - 1, 0, -1):
        path.append(int(backpointers[t, path[-1]]))
    return delta, backpointers, list(reversed(path))

In [14]:
def render_hmm(start_n, nv):
    hmm_states = ["N", "V"]
    hmm_pi = np.array([start_n, 1 - start_n])
    hmm_a = np.array([[1 - nv, nv], [.9, .1]])
    hmm_e = np.array([[.45, .45, .10], [.45, .40, .15]])
    delta, pointers, best = viterbi([0, 1], hmm_pi, hmm_a, hmm_e)
    candidates = [(i, j, hmm_pi[i] * hmm_e[i, 0] * hmm_a[i, j] * hmm_e[j, 1])
                  for i in range(2) for j in range(2)]
    enumerated = max(candidates, key=lambda item: item[2])
    assert np.isclose(np.exp(delta[-1].max()), enumerated[2])
    assert 0 < sum(row[2] for row in candidates) <= 1
    greedy = [int(np.argmax(hmm_pi * hmm_e[:, 0]))]
    greedy.append(int(np.argmax(hmm_a[greedy[0]] * hmm_e[:, 1])))
    greedy_score = hmm_pi[greedy[0]] * hmm_e[greedy[0], 0] * hmm_a[tuple(greedy)] * hmm_e[greedy[1], 1]

    # 两列是两个已知词；节点表示隐藏词性，边只表示状态转移。
    svg = ['<svg viewBox="0 0 720 395" role="img" aria-label="HMM 两步词性状态格图" '
           'style="width:100%;min-width:640px;font-family:system-ui,sans-serif;background:#f8fafc">',
           '<title>Time flies：四条词性路径，绿色为 维特比最优整条路径</title>',
           '<defs><marker id="hmm_arrow" markerWidth="7" markerHeight="7" refX="6" '
           'refY="3" orient="auto"><path d="M0,0 L6,3 L0,6" fill="context-stroke"/></marker></defs>']
    for x, label in [(140, "t = 1：time"), (560, "t = 2：flies")]:
        svg.append(f'<text x="{x}" y="43" text-anchor="middle" fill="#0f172a" font-size="19">{label}</text>')
    label_positions = {(0, 0): (350, 112), (0, 1): (314, 208),
                       (1, 0): (410, 187), (1, 1): (350, 329)}
    for i in range(2):
        for j in range(2):
            y1, y2 = 135 + 170 * i, 135 + 170 * j
            dx, dy = 420, y2 - y1
            length = math.hypot(dx, dy)
            ux, uy = dx / length, dy / length
            color = "#15803d" if [i, j] == best else "#94a3b8"
            width = 4 if [i, j] == best else 2
            svg.append(f'<line x1="{140+32*ux}" y1="{y1+32*uy}" x2="{560-36*ux}" '
                       f'y2="{y2-36*uy}" stroke="{color}" stroke-width="{width}" marker-end="url(#hmm_arrow)"/>')
            lx, ly = label_positions[i, j]
            svg.append(f'<rect x="{lx-47}" y="{ly-17}" width="94" height="25" rx="5" fill="#fff"/>'
                       f'<text x="{lx}" y="{ly}" text-anchor="middle" fill="#334155" font-size="15">'
                       f'{hmm_states[i]}→{hmm_states[j]}：{hmm_a[i,j]:.2f}</text>')
    for t, x in enumerate([140, 560]):
        for i, state in enumerate(hmm_states):
            y = 135 + 170 * i
            active = best[t] == i
            color = "#15803d" if active else "#475569"
            fill = "#dcfce7" if active else "#fff"
            svg.append(f'<circle cx="{x}" cy="{y}" r="30" fill="{fill}" stroke="{color}" stroke-width="3"/>'
                       f'<text x="{x}" y="{y+7}" text-anchor="middle" fill="{color}" font-size="22">{state}</text>'
                       f'<text x="{x}" y="{y+53}" text-anchor="middle" fill="#334155" font-size="14">'
                       f'发射 {hmm_e[i,t]:.2f}</text>')
            if t == 0:
                svg.append(f'<text x="19" y="{y+5}" fill="#334155" font-size="14">π({state})={hmm_pi[i]:.2f}</text>')
    svg.append('<text x="360" y="383" text-anchor="middle" fill="#475569" font-size="14">'
               '绿色：整条最优路径；边标签：转移概率；节点下方：当前词的发射概率</text></svg>')
    display(HTML('<div style="overflow:auto;min-width:0">' + ''.join(svg) + '</div>'))
    table(["完整路径", "P(路径, time flies)", "是否最优"],
          [[hmm_states[i] + " → " + hmm_states[j], f"{p:.6f}", "✓" if np.isclose(p,enumerated[2],rtol=1e-10,atol=1e-12) else ""]
           for i, j, p in candidates], title="枚举四条完整路径（联合概率不是后验概率）")
    table(["时刻 / 已读词", "δ(N)：最佳前缀对数分数", "δ(V)：最佳前缀对数分数", "回指针 N / V"],
          [["1 / time", f"{delta[0,0]:.4f}", f"{delta[0,1]:.4f}", "起点 / 起点"],
           ["2 / flies", f"{delta[1,0]:.4f}", f"{delta[1,1]:.4f}",
            " / ".join(hmm_states[p] for p in pointers[1])]], title="维特比算法：每个终点保留最佳前缀")
    greedy_label = " → ".join(hmm_states[i] for i in greedy)
    best_label = " → ".join(hmm_states[i] for i in best)
    table(["方法", "路径", "联合概率"],
          [["逐步贪心", greedy_label, f"{greedy_score:.6f}"],
           ["维特比整条最优", best_label, f"{enumerated[2]:.6f}"]], title="局部选择与完整路径对比")
    display(HTML(f'<p>观测概率 P(time flies) = {sum(row[2] for row in candidates):.6f}。'
                 '状态转移和发射行分别归一化；所有四路径分数的和不需要等于 1。</p>'))

In [15]:
hmm_ui = panel('hmm', render_hmm, OrderedDict(
    start_n=widgets.FloatSlider(value=.6, min=.1, max=.9, step=.05, description="初始 P(N)", continuous_update=False),
    nv=widgets.FloatSlider(value=.5, min=.1, max=.9, step=.05, description="转移 N→V", continuous_update=False),
))

**本节要点**：马尔可夫链的思想是“下一步只看当前”，用转移矩阵把序列依赖压成可计算的局部形式；N-gram 可看成词上的马尔可夫模型。HMM 把同一思想用在隐藏标签上，再加发射概率连接观测词——N-gram 估“下一字是什么”，HMM 估“这些词对应什么标签”。HMM 的“隐”指标签没有直接观测，不代表它发现了完整语义。标签状态的定义、转移和发射假设仍是建模选择。

**概率怎么学？** $(\pi,A,E)$ 的估计逻辑见 3.2：有标签则计数归一化，无标签则软计数迭代；本算例用手填参数演示解码。

HMM 把序列歧义写成可训练、可推断的概率模型，但离散词表仍需要处理大量没出现过的词与搭配。下一站转回语言建模：能不能让一个词的经验帮助另一个词？


## 4 · 从离散符号到连续表示：通过参数共享经验

**位置**：从查次数到学习表示。**问题**：没见过「小猫趴在」，能不能借用相近上下文的经验？**输入 → 输出**：token ID → 向量 → 下一 token 分布。**与下一节的联系**：固定窗口仍限制可读的历史。

如果每一种词串都是一个互不相干的表格地址，训练数据就很难覆盖所有组合。**约书亚·本吉奥（Yoshua Bengio）**是蒙特利尔大学计算机科学教授、深度学习领域的重要研究者，长期研究表示学习与神经网络。[人物背景](https://yoshuabengio.org/en)

他与合作者在 2003 年的神经概率语言模型中，把词的分布式表示和预测函数一起学习：模型不必只记一个词串出现过几次，还可以在连续表示上共享参数。这是代表性的工作，并非词向量的唯一源头。[原论文](https://jmlr.org/papers/v3/bengio03a.html)

先从神经网络的基本部件看起。**嵌入（embedding）**把离散编号映射为连续向量：编号 $i$ 查到词向量 $e_i=E[i]$；神经元做加权求和并加偏置，再过非线性函数。这里使用 **tanh（hyperbolic tangent，双曲正切函数）**作为非线性函数，再用 **Softmax（归一化指数函数）**把未归一化的分数转换成概率。一个固定窗口语言模型可以写成：

$$h=\tanh(W[e_{t-k};\ldots;e_{t-1}]+b),\quad
\ell=Uh+a,\quad p=\operatorname{softmax}(\ell).$$

$[\ ;\ ]$ 是拼接，$h$ 是上下文表示，$\ell$ 的每一项对应一个候选词的原始分数，称为 **logit（未归一化分数，复数为 logits）**。如果词表有 $V$ 个候选，$E$ 可以看成“词表大小 × 向量维度”，$U$ 把隐藏维度映射回 $V$ 个分数；代码中的 `logits[i]` 就是第 $i$ 个候选的分数。训练会同时改动 $E,W,U$ 等参数；“猫和狗相近”不是预先写入的规则，而是模型可能从共同预测任务中学出的结果。

### 走完一次两字窗口

下面把公式里的查表、拼接、$W$、$\tanh$、$U$ 和 Softmax 都算出来。窗口固定为两个字，候选只有「在、垫、门、天」。$E,W,b,U,a$ 全部手写，用来看清这条计算链，不是训练出来的语义地图。

坐标按例句摆过：猫和狗的第一维都比较大，坐、以及滑块拖近之后的趴，第二维都比较大。滑块只移动「趴」，从远离「坐」到贴着「坐」。$W$ 和 $U$ 始终不变。

**先预测**：窗口取「猫趴」，把趴拖向坐时，「在」的概率会怎样？再把窗口换成「坐在」，「垫」和「门」为什么几乎一样？


In [16]:
window_words = {
    '猫': np.array([1.00, .10]), '狗': np.array([.86, .22]), '坐': np.array([.10, 1.00]),
    '飞': np.array([-.90, -.15]), '在': np.array([.05, -.45]),
}
window_pa_far = np.array([-.20, -.05])
window_pa_near = np.array([.22, .88])
# 第 1 行读「前字 x + 后字 y」：猫/狗的 x、坐/趴的 y 都大，两项相加才高。
# 第 2 行主要读前字的 y。窗口以「坐」开头时这一维升高，垫/门的行对准它。
window_W = np.array([[1.5, 0., 0., 1.5],
                     [0., 1.4, 0., -.6]])
window_b = np.array([-1.5, 0.])
# 垫和门共用同一行：只看见「坐在」时，前面是猫还是狗已经不在窗口里。
window_U = np.array([[3.1, -.3],
                     [-.2, 2.4],
                     [-.2, 2.4],
                     [-2.6, -3.6]])
window_a = np.array([-.15, -.15, -.15, .15])
window_candidates = ['在', '垫', '门', '天']

def window_embeddings(near):
    embeddings = {word: vector.copy() for word, vector in window_words.items()}
    embeddings['趴'] = (1-near)*window_pa_far + near*window_pa_near
    return embeddings

def embedding_distribution(first, second, near):
    embeddings = window_embeddings(near)
    if first not in embeddings or second not in embeddings:
        raise ValueError('窗口里的字需要是猫、狗、坐、趴、飞、在。')
    joined = np.concatenate([embeddings[first], embeddings[second]])
    pre = window_W @ joined + window_b          # 进 tanh 之前的两维数。
    h = np.tanh(pre)
    logits = window_U @ h + window_a            # 每个候选一个未归一化分数。
    exps = np.exp(logits-logits.max())          # 减去最大值只为数值稳定。
    probabilities = exps/exps.sum()
    return embeddings, joined, pre, h, logits, probabilities

In [17]:
def render_embedding(first, second, near):
    embeddings, joined, pre, h, logits, probabilities = embedding_distribution(first, second, near)
    pair = first+second
    count = sum(sentence.count(pair) for sentence in SENTENCES)
    top = int(np.argmax(probabilities))
    note = (f'窗口「{pair}」预测下一个字。6 句例句里它出现 {count} 次；'
            f'当前最高是「{window_candidates[top]}」{probabilities[top]:.2f}。')
    if second == '趴':
        note += '滑块移动「趴」的向量：靠近「坐」时，拼接结果靠近「猫坐」那一类窗口，同一组 W、U 把概率让给「在」。'
    elif pair == '坐在':
        note += '窗口里只剩「坐在」，猫还是狗已经在外面。垫和门两行相同，所以概率平分。'
    elif pair == '飞在':
        note += '「飞在」的 h 落到另一处，「天」那一行和它最齐。'
    else:
        note += '这次窗口没有用到「趴」，滑块不改变这两个字。'
    display(HTML(f'<p style="margin:8px 0;line-height:1.55">{html.escape(note)}</p>'))
    word_order = ['猫', '狗', '坐', '趴', '飞', '在']
    parts = vector_plane([embeddings[w] for w in word_order], word_order,
                         [BLUE, '#3d8fd4', GREEN, ORANGE, '#6d5b8e', '#82909a'],
                         title='词向量；滑块只移动趴', extent=1.6,
                         label_offsets=[(12,-16),(12,18),(-36,-8),(12,16),(-36,-10),(12,16)])
    refs = [('猫坐', '猫', '坐', BLUE), ('坐在', '坐', '在', GREEN), ('飞在', '飞', '在', ORANGE)]
    ref_h = [embedding_distribution(a, b, near)[3] for _, a, b, _ in refs]
    parts += vector_plane([*ref_h, h], [name for name, *_ in refs]+['当前 h'],
                          [color for *_, color in refs]+[RED],
                          left=470, title='当前 h 与三个对照窗口', extent=1.6,
                          label_offsets=[(10,18),(10,-14),(10,16),(12,-20)])
    show_svg(parts, 940, 360, '两字窗口的词向量，以及 tanh 之后的上下文向量')
    vector_grid(np.array([joined]), ['拼接'], '送给 W 的四维：前字 x、前字 y、后字 x、后字 y')
    vector_grid(window_W, ['h 的第 1 维', 'h 的第 2 维'], '同一个 W；偏置 b = (-1.5, 0)')
    table(['步骤', '数值'], [
        [f'e_{first}', embeddings[first].round(3).tolist()],
        [f'e_{second}', embeddings[second].round(3).tolist()],
        ['W·拼接 + b', pre.round(3).tolist()],
        ['h = tanh(...)', h.round(3).tolist()],
    ])
    vector_grid(window_U, window_candidates, '同一个 U：每行对应一个候选；垫和门两行相同')
    p_seen = embedding_distribution('猫', '坐', near)[5][0]
    table(['窗口', 'P(在)'], [['猫坐', p_seen], [pair+'（当前）', probabilities[0]]],
          '和例句里出现过的「猫坐」比一比')
    table(['候选', 'logit = U 的一行 · h + a', 'Softmax 概率'],
          list(zip(window_candidates, logits, probabilities)))
    bars(probabilities, window_candidates, f'窗口「{pair}」的下一字分布；只有这四个候选')

In [18]:
embedding_ui = panel('embedding',render_embedding,OrderedDict(
    first=widgets.Dropdown(options=['猫','狗','坐','飞'],value='猫',description='前一个字'),
    second=widgets.Dropdown(options=['坐','趴','在'],value='趴',description='后一个字'),
    near=widgets.FloatSlider(value=0,min=0,max=1,step=.05,description='趴→坐',continuous_update=False)))

<details><summary>解释与下一步</summary>
趴从远处滑到坐旁边时，变的是查表得到的向量，拼接、$h$ 和每个候选的分数一起变。Softmax 还要求概率和为 1，所以不是只有「在」一个数在动。滑到尽头时，「猫趴」和「猫坐」的 $h$ 靠近，下一字分布也靠近；6 句例句里并没有「猫趴」。

换成「坐在」以后，前一个字已经不在窗口里。例句里「坐在」后面既有垫也有门，这个窗口却看不到前面是猫还是狗。垫和门的两行写成一样，于是概率平分。固定长度窗口仍然读不到更早的字。

这些向量和矩阵如果换一组数，分布就会换一套。训练要做的是按下一字损失去改 $E,W,U$，而不是把坐标轴预先命名成「动物」或「动作」。2003 年这个代表模型仍读取固定长度窗口。能不能让一个状态随着文本向前走，反复更新？为讲清这个问题，我们暂时回看更早出现的循环网络；这是概念的衔接，不是年代倒置后的发明顺序。
</details>


## 5 · 循环神经网络：让状态随输入逐步更新

**位置**：输入长度不再固定为一个短窗口。**问题**：读到第十个词时，怎样利用前九个词？**输入 → 输出**：当前向量与旧状态 → 新状态与候选分布。**与下一节的联系**：状态压缩、长程学习与顺序计算。

**循环神经网络（Recurrent Neural Network，RNN）**用循环连接，让先前状态参与当前计算。

**杰弗里·洛克·埃尔曼（Jeffrey Locke Elman，1948—2018）**是加利福尼亚大学圣迭戈分校的认知科学家，研究语言学习与处理。他在 1990 年发表的简单循环网络工作，用上下文单元保留先前状态，为研究序列中的结构提供了一个代表模型。[人物背景](https://crl.ucsd.edu/elman/obituary/)

这种结构改变了保存历史的方式：不必为每一种长度重新建一个网络，同一套参数可以逐步读过不同长度的序列。这里说的“记忆”是随输入变化的状态，不是训练时缓慢改变的参数。[Elman，Finding Structure in Time](https://doi.org/10.1207/S15516709COG1402_1)

$$h_0=0,\qquad h_t=\tanh(\underbrace{W_x e_t}_{当前输入}+\underbrace{W_h h_{t-1}}_{过去状态}+b),\qquad
p(x_{t+1}\mid x_{\leq t})=\operatorname{softmax}(Uh_t+a).$$

$e_t$ 是当前字符的向量，$h_t$ 是这一次输入产生的状态，不是模型参数。相同的 $W_x,W_h$ 在每一步复用：先把“当前输入”和“过去状态”各自变换后相加，再用 tanh 得到新状态。$\tanh$ 把每维压进 $(-1,1)$，输入很大或很小时趋于饱和。$h_t$ 的维度固定，内容随历史改变；这不等于它只能记住最后一个词。但要把许多关系压进状态、通过很长的链学习早期影响，并不容易。计算 $h_t$ 也需要先得到 $h_{t-1}$。

### 5.1 先展开四步，看看“记忆”究竟是什么

**先预测**：两句最后一个字都是「在」，最后状态一定相同吗？改变最前面的猫/狗，再逐步移动到第 4 步，追踪两维状态。权重和字符输入均为手工设定的算例参数，下一字分布也只是三候选演示。


In [19]:
# 窗口演示用的是另一组 W、U。这里单独留一张手写输出矩阵，把二维状态映到三个候选。
geometry_U = np.array([[2.,-.5],[-.5,2.],[-1.,1.]])
geometry_candidates = ['垫','门','天']

rnn_input_vectors = {'小':[.2,.1],'猫':[1.,0.],'狗':[0.,1.],'坐':[.1,.3],'在':[.2,-.1]}
rnn_Wx = np.array([[.8,-.2],[.1,.7]])
rnn_Wh = np.array([[.65,.15],[-.1,.7]])

def rnn_states(prefix):
    # states[0] 就是 h_0；states[t] 是读完前 t 个字符后的状态。
    states = [np.zeros(2)]
    for ch in prefix:
        x = np.array(rnn_input_vectors[ch])
        # 对应 h_t = tanh(W_x e_t + W_h h_{t-1})。
        input_part = rnn_Wx @ x
        memory_part = rnn_Wh @ states[-1]
        states.append(np.tanh(input_part + memory_part))
    return np.array(states)

In [20]:
def render_rnn(prefix, step):
    states = rnn_states(prefix)
    parts=[]
    for t in range(5):
        x,y=65+180*t,100
        color = RED if t==step else BLUE
        if t:
            parts.append(svg_arrow(x-130,y,x-38,y,'#6d8595'))
            parts += [svg_text(x,38,prefix[t-1],ORANGE,20),svg_arrow(x,47,x,70,ORANGE)]
        parts += [f'<rect x="{x-36}" y="{y-25}" width="72" height="50" rx="4" fill="#eef4f8" stroke="{color}"/>',
                  svg_text(x,y+5,f'h{t}',color,18),svg_text(x,158,f'({states[t,0]:.2f}, {states[t,1]:.2f})',size=12)]
    show_svg(parts,900,195,'同一组循环参数在四步中复用；状态沿箭头传递')
    h_prev, x = states[step-1],np.array(rnn_input_vectors[prefix[step-1]])
    table(['本步计算','两维数值'],[
        ['输入字符',prefix[step-1]],['输入贡献 Wx · e', (rnn_Wx@x).round(4).tolist()],
        ['记忆贡献 Wh · h旧',(rnn_Wh@h_prev).round(4).tolist()],['tanh 后的新状态',states[step].round(4).tolist()]])
    logits = geometry_U@states[step]
    probs = np.exp(logits-logits.max()); probs /= probs.sum()
    bars(probs,geometry_candidates,'手工循环网络：当前状态 → 三候选下一字分布')

In [21]:
rnn_ui = panel('rnn',render_rnn,OrderedDict(
    prefix=widgets.Dropdown(options=['小猫坐在','小狗坐在'],value=PREFIX,description='前缀'),
    step=widgets.IntSlider(value=4,min=1,max=4,description='读到第几字',continuous_update=False)))

### 5.2 一条很长的链，为什么难学？从梯度连乘到记忆门

假设我们想让最后一次预测的错误，改变很早输入的处理方式。反向传播必须沿状态链传回来。为看清连乘，先把 RNN 缩成标量，首个输入进入 $h_0=s$，以后没有新输入：

$$h_t=\tanh(rh_{t-1}),\qquad
\frac{\partial h_T}{\partial s}=\prod_{t=1}^T r(1-h_t^2).$$

这里的导数表示：只把最初的 $s$ 改动一点，最终状态 $h_T$ 对它有多敏感。小于 1 的因子反复相乘会衰减，大于 1 的因子可能放大；tanh 饱和又会压低导数。实际 RNN 的导数是矩阵乘积，下面只隔离一种机制，不能用这个标量图完整评价某个训练好的网络。

**长短期记忆网络（Long Short-Term Memory，LSTM）**针对长时间间隔中的误差信号传播问题设计记忆单元，代表论文发表于 1997 年。

两位共同提出者是**塞普·霍赫赖特（Sepp Hochreiter）**与**于尔根·施密德胡贝尔（Jürgen Schmidhuber）**。霍赫赖特长期研究梯度消失、循环网络与表示学习；施密德胡贝尔长期研究循环网络、深度学习与学习算法。二人的合作把“如何让误差信号跨越长时间间隔”落实为可训练的网络结构。[霍赫赖特的研究背景](https://www.jku.at/en/lit-artificial-intelligence-lab/about-us/team/)、[施密德胡贝尔的研究背景](https://people.idsia.ch/~juergen/life/)

现代常见写法中的记忆更新是：

$$c_t=f_t\odot c_{t-1}+i_t\odot g_t,\quad h_t=o_t\odot\tanh(c_t).$$

遗忘门 $f_t$ 控制保留多少，输入门 $i_t$ 控制写入多少，输出门 $o_t$ 控制读出多少；门通常由当前输入和旧状态一起学习。下面只保留无新写入的一条记忆通路 $c_t=fc_{t-1}$，其局部保留系数连乘为 $f^T$。固定 $f$ 接近 1 有利于保留这条通路，完整网络的总导数还存在其他路径；不能推出所有长依赖都得到解决。原始 1997 单元与此处的现代常见写法也不是逐项一致。[LSTM 原论文](https://www.bioinf.jku.at/publications/older/2604.pdf)

**先预测**：间隔从 2 变为 20，$r=0.7$ 的敏感度会怎样？把 $f$ 从 0.8 调到 0.99，图中的记忆通路发生什么变化？


In [22]:
def memory_curves(gain, forget, delay):
    # signal 是最早的输入 s；requires_grad 让 PyTorch 追踪 dh_t/ds。
    signal = torch.tensor(.2,requires_grad=True)
    h = signal
    sensitivities = [1.]
    for t in range(delay):
        h = torch.tanh(gain*h)
        # 每一步都求相对于最初 signal 的导数，而非某个参数的梯度。
        sensitivities.append(float(torch.autograd.grad(h,signal,retain_graph=True)[0]))
    # 简化记忆通路 c_t=f*c_{t-1} 的解析结果就是 f^t。
    retained = forget**np.arange(delay+1)
    return np.array(sensitivities),retained

In [23]:
def render_memory(gain, forget, delay):
    sensitivities,retained = memory_curves(gain,forget,delay)
    # 对数轴显式标记；极小值仅在显示时截为 10^-8，不改计算结果。
    def point(t,value):
        return 80+t/delay*700,230-(np.clip(np.log10(max(abs(value),1e-8)),-8,2)+8)/10*180
    parts=[]
    for exponent in [-8,-6,-4,-2,0,2]:
        _,y=point(0,10.**exponent)
        parts += [f'<path d="M80 {y} H800" stroke="#e0e7eb"/>',svg_text(68,y+4,f'10^{exponent}',size=12,anchor='end')]
    for values,color,label in [(sensitivities,BLUE,'RNN: |∂h_t/∂s|'),(retained,GREEN,'固定门通路: f^t')]:
        points=' '.join(f'{x:.2f},{y:.2f}' for x,y in [point(t,v) for t,v in enumerate(values)])
        parts.append(f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="3"/>')
        parts.append(svg_text(90,285 if color==BLUE else 310,label,color,14,anchor='start'))
    parts += [svg_text(80,254,0,size=12),svg_text(800,254,delay,size=12),
              svg_text(815,272,'间隔 t',size=12),svg_text(80,25,'敏感度 / 保留系数（纵轴对数）',size=15,anchor='start')]
    show_svg(parts,900,335,'RNN 的实际自动微分敏感度与固定门记忆通路的比较')
    table(['间隔 T','RNN 实际自动微分','固定门通路 f^T'],[[delay,f'{sensitivities[-1]:.8g}',f'{retained[-1]:.8g}']])
    print('这是手工标量动力系统。门不是注意力权重，不需要在多个位置之间相加为 1。')

In [24]:
memory_ui = panel('memory',render_memory,OrderedDict(
    gain=widgets.FloatSlider(value=.7,min=.2,max=1.4,step=.05,description='循环系数 r',continuous_update=False),
    forget=widgets.FloatSlider(value=.95,min=.5,max=1,step=.01,description='保留门 f',continuous_update=False),
    delay=widgets.IntSlider(value=16,min=2,max=24,description='间隔 T',continuous_update=False)))

<details><summary>解释与下一步</summary>
代码中的 `states[0]` 是 $h_0$，每循环一次就得到一个新状态。猫和狗进入同一组参数，却可以留下不同的状态；RNN 没有被限制为只看最后一个字。标量实验中的梯度随间隔显著变化，LSTM 风格的保留通路提供了另一种传播方式。但循环依赖仍然存在；若翻译器只把源句最后状态交给解码器，源句的许多细节还要挤在同一个接口里。下一站的办法是保留各个源位置，在生成需要时重新读取。
</details>


## 6 · 2014：翻译长句时，能不能回头看？

**位置**：从一个序列映射到另一个序列。**问题**：生成译文不同词时，怎样获取不同的源句信息？**输入 → 输出**：当前解码状态与全部源位置表示 → 本步上下文向量。**与下一节的联系**：已有读取机制，是否还必须让循环承担主干？

把统计思想接到翻译上，并不是神经网络时代才开始。**沃伦·韦弗（Warren Weaver，1894—1978）**是美国数学家和科研管理者，曾长期负责洛克菲勒基金会的自然科学项目。他在 1949 年的《翻译》备忘录中，把上下文、密码分析与统计思想引入机器翻译的设想；语言研究由此也与通信和计算中的问题相互启发。那还不是可运行的神经翻译系统，却标出了“翻译可以当作信息处理问题”这条线。[人物背景](https://www.rockefellerfoundation.org/wp-content/uploads/Beyond-Charity.pdf.pdf)、[备忘录原文](https://www.mt-archive.net/50/Weaver-1949.pdf)

早期的**编码器—解码器（Encoder–Decoder）**神经翻译模型常先读完整句，把它压成固定长度向量，再据此生成目标句。句子越长、细节越多，这个接口越难承担所有信息。

**德米特里·巴赫达瑙（Dzmitry Bahdanau）**是研究自然语言理解与神经机器翻译的研究者，曾在蒙特利尔大学和魁北克人工智能研究机构攻读博士。[人物背景](https://mila.quebec/en/directory/dzmitry-bahdanau)

他与合作者在 2014 年公开、2015 年会议发表的工作让解码器在每次生成时，对源位置做软对齐：写到不同目标词时，可以从不同的源状态取回内容。**注意力机制（attention，后文写作 Attention）**在这里已经出现，早于 Transformer。[原论文](https://arxiv.org/abs/1409.0473)

$$e_{tj}=\operatorname{score}(s_{t-1},h_j),\quad
a_{tj}=\frac{\exp(e_{tj})}{\sum_k\exp(e_{tk})},\quad
c_t=\sum_j a_{tj}h_j.$$

$s_{t-1}$ 是解码器旧状态，$h_j$ 是源句第 $j$ 个表示；$c_t$ 是本步读出的上下文向量，仍不是一个被选中的字。它会再与解码器计算连接，产生词表分数。原始 Bahdanau 模型使用加性评分；下面**为解释几何关系采用简化的点积算例**，其中的评分方法与原论文的加性评分不同。

在这个算例中，当前需要的信息用**查询（Query，记作 q）**表示，源位置提供用于匹配的**键（Key，记作 k）**和传递内容的**值（Value，记作 v）**。把查询画成向量 $q$，键画成 $k_j$。点积满足 $q^\top k=\|q\|\|k\|\cos\theta$：方向和长度一起决定分数。左图转动查询，右图把加权值向量首尾相接；连线越粗，权重越大。

**先预测**：查询更靠近某个键时，输出一定等于那个位置的值向量吗？加权求和与直接选择一个位置，有什么区别？


In [25]:
cross_keys = np.array([[1.5,0.],[0.,1.5],[-1.,-.8]])
cross_values = np.array([[1.,.1],[.1,1.],[-.8,-.4]])
cross_labels = ['小猫','坐在','垫子上']

def cross_attention(angle):
    radians = math.radians(angle)
    query = 1.5*np.array([math.cos(radians),math.sin(radians)])
    # 第一步：query 与每个 key 点积，得到匹配分数。
    scores = cross_keys@query/math.sqrt(2)
    # 第二步：Softmax 把分数变成和为 1 的读取权重。
    exps = np.exp(scores-scores.max())
    weights = exps/exps.sum()
    # 第三步：用权重混合二维 Value，得到上下文向量 c。
    context = weights@cross_values
    return query,scores,weights,context

In [26]:
def render_cross_attention(angle):
    query,scores,weights,context = cross_attention(angle)
    weighted_geometry(query,cross_keys,cross_values,weights,cross_labels)
    parts=[]
    for j,label in enumerate(cross_labels):
        x=140+j*300
        parts += [svg_text(x,30,label,BLUE,18),
                  svg_arrow(x,45,440,137,BLUE,1+weights[j]*9,.25+.75*weights[j]),
                  svg_text(x,78,f'a{j} = {weights[j]:.3f}',size=13)]
    parts.append(svg_text(440,170,'本步上下文 c，继续送给解码器',GREEN,17))
    show_svg(parts,900,200,'源句位置按权重连向本步上下文向量')
    table(['源位置','点积分数 / √2','权重','二维 V'],[
        [label,scores[j],weights[j],cross_values[j].tolist()] for j,label in enumerate(cross_labels)])
    print('加总结果 c =',context.round(4),'；不是译文候选的概率。')

In [27]:
cross_ui = panel('cross_attention',render_cross_attention,OrderedDict(
    angle=widgets.IntSlider(value=0,min=0,max=360,step=5,description='查询角度',continuous_update=False)))

<details><summary>解释与下一步</summary>
标准 Softmax 给多个位置非零权重，输出是多个值向量的混合；点积只负责产生评分，后面还需要归一化与加权求和。这个算例没有训练翻译器，也没有证明某条注意力连线就是模型的因果解释。它只说明“当前状态按需读取一组位置”怎样运作。

到此为止，读源句和生成译文的主干仍可使用 RNN。2017 年的下一步，是让 attention 与逐位置网络直接承担主干中的信息处理。
</details>


## 7 · 2017：把读取放进主干，Transformer 出场

**位置**：从带 Attention 的 RNN 到 Transformer。**问题**：能否减少位置间逐步递推，把已知序列同时送入计算？**输入 → 输出**：整排带位置线索的向量 → 整排更新后的表示。**与下一节的联系**：怎样用这种表示学习可迁移的语言能力？

**阿希什·瓦斯瓦尼（Ashish Vaswani）**是神经网络与机器翻译研究者，发表这项工作时在谷歌大脑团队工作，也是 Transformer 原论文的共同作者。

他与合作者在 2017 年的《Attention Is All You Need》（可译为《只需注意力机制》）中提出用 attention 等组件构成序列转换主干，不再依靠循环和卷积作为主干机制。原论文是 **Encoder–Decoder 翻译模型**，不是 GPT。Encoder 内的位置互相读取；Decoder 一边读取已经生成的前缀，一边通过交叉注意力读取源句。[Vaswani 等，2017](https://arxiv.org/abs/1706.03762)

与 RNN 相隔很远的信息要逐步传递不同，自注意力可以在一层里直接让两个位置相连。训练时，输入序列已知，各位置的向量更新可在层内并行计算；这不意味着生成整句只需要一步。标准全注意力还要计算大约 $T^2$ 个位置对，长上下文会带来成本。

接下来逐步分析 **GPT 风格的因果模块（causal Transformer block，后文简称 Block）**。它只允许各位置读取自身及此前的输入；归一化、激活函数和位置向量的具体配置将在后文说明。这个实验模型用于理解生成式架构，与 2017 年原版的编码器—解码器结构有所不同。我们先补上位置顺序，再展开查询、键和值的计算，最后解释逐位置前馈网络、残差连接与归一化。


In [28]:
def render_paths(distance):
    parts=[]
    for row,title in [(0,'RNN：信息沿相邻状态逐步传递'),(1,'一层自注意力：可直接读取可见的远处位置')]:
        y=85+row*120
        parts.append(svg_text(40,y-40,title,size=15,anchor='start'))
        for t in range(distance+1):
            x=65+t/distance*730
            parts += [f'<circle cx="{x}" cy="{y}" r="12" fill="{BLUE if t in [0,distance] else "#dde7ed"}"/>']
            if row==0 and t:
                prev=65+(t-1)/distance*730
                parts.append(svg_arrow(prev+13,y,x-13,y,'#718997',1.5))
        if row:
            parts.append(f'<path d="M795 {y} Q430 {y-65} 65 {y}" fill="none" stroke="{GREEN}" stroke-width="3"/>')
    show_svg(parts,900,260,'相同位置间隔下，RNN 与自注意力的信息路径')
    table(['比较对象','本图中跨越两端的路径','须注意的代价'],[
        ['RNN',f'{distance} 次相邻状态传递','顺序依赖；单步还需矩阵计算'],
        ['自注意力','同一层一次读取',f'若长度 T={distance+1}，完整评分表有 {(distance+1)**2} 项']])

In [29]:
paths_ui = panel('paths',render_paths,OrderedDict(
    distance=widgets.IntSlider(value=6,min=2,max=12,description='位置间隔',continuous_update=False)))

### 7.1 同一个字在不同位置，为什么还要加一个向量？

单独查词表时，同一个 ID 总会取到同一行。输入中的「在」还没读过前面的「小猫」，也没有自己的位置标记。为此我们再查一张**位置表**，把位置向量与字符向量相加：

$$x_t = E[\mathrm{id}_t] + P[t]$$

`E` 按词表 ID 查，`P` 按位置查；相加后仍是一排同样长的数。这个模型的两张表都由训练学习。位置标记告诉后面的计算“这行来自第几位”，上下文信息则要等 Attention 来汇集。

下面使用四维**手工算例**。它只方便我们看数，不是学好的语义向量，也不是后面 64 维模型的参数。


In [30]:
toy_rng = np.random.default_rng(12)
toy_E = toy_rng.normal(0, .3, (VOCAB_SIZE, 4))
# 贯穿前缀的四个字符使用便于计算的数值。
toy_E[[STOI[ch] for ch in PREFIX]] = np.array([
    [1.,0.,0.,0.], [0.,1.,0.,0.], [0.,0.,1.,0.], [1.,1.,0.,0.]])
toy_P = np.array([[0.,0.,0.,0.], [.1,0.,0.,0.], [0.,.1,0.,0.], [0.,0.,.1,0.]])
toy_tokens = toy_E[prefix_ids[0].numpy()]
toy_X = toy_tokens + toy_P
vector_grid(toy_tokens, list(PREFIX), '手工字符向量 E[id]')
vector_grid(toy_P, [f'位置 {i}' for i in range(4)], '手工位置向量 P[t]')
vector_grid(toy_X, list(PREFIX), '相加后，作为 Block 的输入 X')
print('X 的形状：', toy_X.shape, '=（位置数 T，向量维度 D）')


小,1.000,0.000,0.000,0.000
猫,0.000,1.000,0.000,0.000
坐,0.000,0.000,1.000,0.000
在,1.000,1.000,0.000,0.000


位置 0,0.000,0.000,0.000,0.000
位置 1,0.100,0.000,0.000,0.000
位置 2,0.000,0.100,0.000,0.000
位置 3,0.000,0.000,0.100,0.000


小,1.000,0.000,0.000,0.000
猫,0.100,1.000,0.000,0.000
坐,0.000,0.100,1.000,0.000
在,1.000,1.000,0.100,0.000


X 的形状： (4, 4) =（位置数 T，向量维度 D）


In [31]:
def inspect_toy_input(position):
    ch = PREFIX[position]
    table(['步骤', '对应内容'], [
        ['当前位置 / ID', f'{position} / {ch!r} / {STOI[ch]}'],
        ['字符向量', toy_tokens[position].round(3).tolist()],
        ['位置向量', toy_P[position].round(3).tolist()],
        ['相加后的输入', toy_X[position].round(3).tolist()],
    ])

In [32]:
input_ui = panel('input', inspect_toy_input, OrderedDict(
    position=widgets.Dropdown(options=[(f'{i}：{ch}',i) for i,ch in enumerate(PREFIX)], value=3, description='查看位置')))

**先预测**：如果把「猫」挪到另一位置，它查到的字符向量会变吗？相加后的输入向量呢？

<details><summary>核对与下一步</summary>字符 ID 没变，字符向量不变；位置变了，位置向量通常不同，因此相加后的输入会变。这仍没有读到其他字符。接下来要解决的是：怎样让当前位置的信息和前文发生关系？</details>

<details><summary>选读：独热编码与位置的数学解释</summary>
**独热编码（one-hot encoding）**是只有对应 ID 那一项为 1、其余项为 0 的向量。独热向量乘嵌入表，等价于直接取出那一行；实际实现用查表避免构造大量的 0。没有位置输入、没有因果掩码的全自注意力具有置换等变性：重排输入各行，输出也随之重排，并非输出完全不变。GPT 的因果掩码也提供了访问结构限制；我们仍显式学习位置向量，帮助模型利用位置。
</details>


### 7.2 Attention：把“别的位置有什么”写入当前位置

**位置**：进入第一个 Block，先处理位置之间的信息流。**问题**：「小猫坐在」和「小狗坐在」里的「在」，输入字符相同、位置相同，如果一直逐位置独立计算，就无法根据前面的动物调整下一字预测。

我们需要一次跨位置的信息汇集：为当前位置分配一组权重，决定从哪些位置各取多少内容，再把这些内容相加。**Attention 的输出仍然是向量，不是一个被选中的词，也不是答案文本。**

先暂时指定一组权重，理解“取回来”这一步。我们把每个位置提供的内容记作 `V`（Value，值）。本算例从四维输入投影出两维内容；每个位置用同一个矩阵变换，但它们的输入不同。


In [33]:
roadmap({2})
toy_WV = np.array([[1.,0.], [0.,1.], [1.,1.], [-1.,1.]])
toy_V = toy_X @ toy_WV
hand_weights = np.array([.1, .6, .1, .2])   # 暂时人为分配给最后位置「在」
hand_mixed = hand_weights @ toy_V
vector_grid(toy_V, list(PREFIX), '同一个输入 X 变换成每个位置提供的内容 V')
table(['来源', '给它的权重', 'V 的第 0 维', 'V 的第 1 维', '贡献第 0 维', '贡献第 1 维'],
      [(ch, float(w), *v.tolist(), *(w*v).tolist()) for ch,w,v in zip(PREFIX,hand_weights,toy_V)])
print('汇集结果 = 0.1 V[小] + 0.6 V[猫] + 0.1 V[坐] + 0.2 V[在] =', hand_mixed)


小,1.000,0.000
猫,0.100,1.000
坐,1.000,1.100
在,1.100,1.100


小,0.100,1.000,0.000,0.100,0.000
猫,0.600,0.100,1.000,0.060,0.600
坐,0.100,1.000,1.100,0.100,0.110
在,0.200,1.100,1.100,0.220,0.220


汇集结果 = 0.1 V[小] + 0.6 V[猫] + 0.1 V[坐] + 0.2 V[在] = [0.48 0.93]


#### 从加权平均到 Q/K/V：把“匹配”和“内容”分开

刚才权重由我们填写。模型需要根据输入自己计算权重，因此从同一个输入矩阵 `X` 产生三组向量：

$$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V$$

- **Q（Query，查询）**：当前位置拿什么特征去匹配其他位置。
- **K（Key，键）**：每个位置拿什么特征供别人匹配。
- **V（Value，值）**：匹配之后，每个位置实际贡献什么内容。

这里的 V 表示值，与第 3 节词性标记中代表动词的 V 含义不同。三个矩阵是可以学习的参数，并非人为给各个字填写“问题、标签、答案”。同一个字在后续层中已经包含上下文，这三组向量也会相应改变。

对最后位置「在」，先拿它的 Q 与每个位置的 K 做点积：两个向量对应坐标相乘再求和。**分数用于决定取多少，V 决定取回什么**。分开投影，让匹配用的特征不必和传递内容的特征完全相同。投影可学习，点积的数值如何对应有效的匹配关系，需要通过训练学习。


In [34]:
toy_WQ = np.array([[1.,0.], [0.,1.], [1.,0.], [0.,1.]])
toy_WK = np.array([[1.,0.], [0.,1.], [0.,1.], [1.,0.]])
toy_Q = toy_X @ toy_WQ
toy_K = toy_X @ toy_WK
# 所有位置同时匹配：第 i 行、第 j 列 = Q[i] 与 K[j] 的点积。
toy_raw_scores = toy_Q @ toy_K.T
toy_scores = toy_raw_scores / math.sqrt(toy_Q.shape[-1])
vector_grid(toy_Q, list(PREFIX), 'Q：每个位置用于查询的向量')
vector_grid(toy_K, list(PREFIX), 'K：每个位置供匹配的向量')
table(['被查询的位置', 'Q[在] · K[j]', '除以 √2 后'],
      [(ch, toy_raw_scores[3,j], toy_scores[3,j]) for j,ch in enumerate(PREFIX)],
      f'Q[在] = {toy_Q[3].tolist()}；分数仍可为负，也不要求和为 1')


小,1.000,0.000
猫,0.100,1.000
坐,1.000,0.100
在,1.100,1.000


小,1.000,0.000
猫,0.100,1.000
坐,0.000,1.100
在,1.000,1.100


小,1.100,0.778
猫,1.110,0.785
坐,1.100,0.778
在,2.200,1.556


#### 把匹配分数变成权重：Softmax 与因果掩码

我们希望对 V 做加权平均，因此需要非负、总和为 1 的权重。标准 Transformer 用 Softmax：先对每个分数取指数，再除以指数总和。

$$a_{tj}=\frac{\exp(s_{tj})}{\sum_{j'}\exp(s_{tj'})},\qquad z_t=\sum_j a_{tj}v_j$$

例如分数 `(2, 1, 0)` 变成约 `(0.665, 0.245, 0.090)`。高分获得更多权重，其他位置仍可贡献内容；这种平滑计算也便于训练。这里 Softmax 的选项是**输入位置**，不是词表里的字符。

**因果掩码（causal mask，后文写作 Mask）**用来限制可读取的范围：**位置 t 的输出预测 t+1，只能读取位置 ≤ t 的输入**。训练时整句文本已经在手中，如果「坐」能读到后面的「在」，下一字预测就会直接读取到目标答案，造成训练信息泄漏。因此，在归一化之前把未来位置的分数设为负无穷；指数变成 0，这些位置便不参与加权。

完整的一头注意力是：

$$Z=\operatorname{softmax}_{\text{每行}}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V$$

`M` 在允许读取的位置为 0，未来位置为负无穷。除以 `√d_k` 是为了控制点积分数随维度增加而扩大的典型尺度，减少 Softmax 过早变得极尖的情况；它不是概率归一化。


In [35]:
def softmax_np(scores):
    # 每行减最大值：指数的公共倍数会在归一化中消掉，结果不变且更稳定。
    shifted = scores - scores.max(axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / exps.sum(axis=-1, keepdims=True)

def toy_attention(query=3, key=1, delta=0., causal=True):
    # scores[i, j] 表示第 i 个位置想读取第 j 个位置的匹配程度。
    scores = toy_scores.copy()
    scores[query, key] += delta  # 控制实验：只改变指定的一条匹配分数。
    if causal:
        # 上三角是未来位置；-inf 经过 Softmax 后权重为 0。
        scores = np.where(np.tril(np.ones_like(scores, dtype=bool)), scores, -np.inf)
    # 每一行独立归一化：一行就是一个 query 对所有来源位置的权重。
    weights = softmax_np(scores)
    # 加权求和的是 V，输出仍是向量，不是词表概率。
    mixed = weights @ toy_V
    return scores, weights, mixed

scores, weights, mixed = toy_attention()
print('softmax(2,1,0) =', softmax_np(np.array([[2.,1.,0.]]))[0])
vector_grid(weights, list(PREFIX), '行是查询位置，列依次是 小 / 猫 / 坐 / 在；每行和为 1')
vector_grid(mixed, list(PREFIX), '每行是该位置取回的两维内容，仍然不是字符')
assert np.allclose(weights.sum(axis=-1), 1.)
assert np.all(weights[np.triu_indices(4, 1)] == 0.)


softmax(2,1,0) = [0.665 0.245 0.09 ]


小,1.000,0.000,0.000,0.000
猫,0.344,0.656,0.000,0.000
坐,0.476,0.270,0.254,0.000
在,0.193,0.194,0.193,0.420


小,1.000,0.000
猫,0.410,0.656
坐,0.757,0.549
在,0.867,0.868


#### 连续实验：修改一条匹配分数，取回的内容怎样变化？

先选查询位置，再选一条来源位置，提高或降低它的分数。观察同一张表里的分数、权重、V 和实际贡献，最后看加总结果。

**先预测**：选「坐」作为查询位置，选未来的「在」作为来源。保持因果 Mask 开启，即使把这条分数提高 3，输出会变吗？再关闭 Mask 会怎样？最后一行的查询位置没有未来位置，开关 Mask 不会改变它。


In [36]:
def render_attention(query, key, delta, causal):
    scores, weights, mixed = toy_attention(query, key, delta, causal)
    table(['来源位置', '归一化前分数', '权重', 'V0', 'V1', '权重 × V0', '权重 × V1'], [
        [f'{j}：{ch}', '被屏蔽' if not np.isfinite(scores[query,j]) else scores[query,j],
         weights[query,j], *toy_V[j].tolist(), *(weights[query,j]*toy_V[j]).tolist()]
        for j,ch in enumerate(PREFIX)])
    bars(weights[query], [f'{j}：{ch}' for j,ch in enumerate(PREFIX)], f'位置 {query}「{PREFIX[query]}」给输入位置的权重')
    weighted_geometry(toy_Q[query],toy_K,toy_V,weights[query],list(PREFIX))
    print('把表中最后两列分别相加，得到：', mixed[query].round(4))
    print('这些数将送进后续投影和残差；它们不是下一个字符的概率。')
    if not causal:
        print('当前关闭 Mask，仅用于对照；后面的完整 GPT 模型始终启用因果 Mask。')

token_options = [(f'{i}：{ch}',i) for i,ch in enumerate(PREFIX)]

In [37]:
attention_ui = panel('attention', render_attention, OrderedDict(
    query=widgets.Dropdown(options=token_options, value=3, description='谁在查询'),
    key=widgets.Dropdown(options=token_options, value=1, description='改谁的分数'),
    delta=widgets.FloatSlider(value=0, min=-3, max=3, step=.25, description='分数增量', continuous_update=False),
    causal=widgets.Checkbox(value=True, description='开启因果 Mask')))

<details><summary>核对与下一步</summary>先看代码的三步：`scores` 是匹配分数，`weights` 是 Softmax 权重，`mixed = weights @ V` 是取回的向量。未来位置先被掩码屏蔽，分数怎么调都没有贡献。关闭 Mask 后，这条来源才能参与归一化。提高一个可见位置的分数，会改变整行权重，而非只改一个权重。现在我们得到了每个位置汇集来的内容；下一步解释它怎样写回原表示，以及为什么还需要逐位置前馈网络。</details>

<details><summary>选读：这里的简化有什么边界？</summary>
手工投影没有训练，不应从它的热图推断猫和垫子的真实语义关系。权重反映这一层这一头的混合比例，也不等于完整模型预测的因果解释。Softmax 是标准方案，不代表所有注意力变体都必须采用它。点积缩放的直觉基于各分量尺度近似稳定等条件，不是对任意训练状态的严格保证。
</details>


### 7.3 组成 Block：交换信息、加工特征，并把更新写回

**位置**：从一头 Attention 走向完整 Transformer Block。**问题**：刚取回的是一份更新，还要表达多种匹配方式，并让模型逐步加工信息。

#### 多头：同一份输入，使用多组投影

一头用一组 Q/K/V 投影；多头让多组投影分别算注意力，再拼接并投影回原来的向量维度。每头读完整输入的投影，不是简单把原输入切成几块，也没有预先指定“第一头学语法”。不同偏好可能从训练中产生。

后面的真实模型使用 `D=64`、`H=4`，每头 `d_k=16`。手算中一头只保留两维；真实模型中每头取回 16 维内容，四头拼成 64 维，再经输出投影得到 64 维更新，因此能加回原表示。下面的代码不调用 `nn.Transformer`，直接实现刚才的公式。


In [38]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, capture=False):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # (B,T,D) 拆成 (B,H,T,d_k)，每个头独立算一张读取表。
        q, k, v = [a.reshape(B,T,self.n_head,self.head_dim).transpose(1,2) for a in (q,k,v)]
        # QK^T 的形状是 (B,H,T,T)：行是 query，列是来源位置。
        scores = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        # tril 保留来源位置 <= query 位置；(T,T) 会广播到所有头和样本。
        allowed = torch.ones(T,T,device=x.device,dtype=torch.bool).tril()
        masked_scores = scores.masked_fill(~allowed, -torch.inf)
        # 对最后一个来源位置维度做 Softmax，得到每行读取权重。
        weights = F.softmax(masked_scores, dim=-1)
        # 权重混合 V，拼回各头，再投影回 D 维。
        mixed = weights @ v
        joined = mixed.transpose(1,2).contiguous().reshape(B,T,D)
        update = self.proj(joined)
        trace = dict(q=q,k=k,v=v,scores=scores,masked_scores=masked_scores,
                     weights=weights,mixed=mixed,joined=joined,update=update) if capture else None
        return update, trace


#### 前馈网络：汇集上下文之后，在各位置加工特征

Attention 负责位置之间的信息交换。**FFN（Feed-Forward Network，前馈网络）**独立处理每个位置的向量；这里采用 **MLP（Multilayer Perceptron，多层感知机）**结构：所有位置用同一套参数，但各自的输入已包含不同上下文。

其中 **GELU（Gaussian Error Linear Unit，高斯误差线性单元）**是本模型使用的激活函数：

$$\operatorname{FFN}(h)=W_2\,\operatorname{GELU}(W_1h+b_1)+b_2$$

这是一种按列向量书写的数学记法；PyTorch 用最后一维保存特征，完成相同的变换。一个神经元就是先算加权和再过激活函数，许多神经元组成一层。这里先从 64 维扩到 256 维，经非线性，再回到 64 维。非线性让它不只是两次矩阵乘法合成的一次线性变换。

FFN 不直接查询别的位置，但能加工 Attention 刚取来的组合特征。它和 Attention 都学习参数，不能把 Transformer 仅仅理解成“算一张注意力表”。

#### 残差与层归一化：保留原表示，并控制送入分支的尺度

**残差**把原向量与分支更新相加：`新表示 = 原表示 + 更新`。分支学习的是怎样补充或修改现有表示；这条直接通路也有助于深层训练，不保证每一维的信息都原样保留。

**层归一化（Layer Normalization，LN，代码中写作 LayerNorm）**在每个位置的特征维度上归一化，再用可学习的缩放和偏移调整结果。它不跨位置平均，也不把句子变成概率。

本模型采用 **Pre-LN（Pre-Layer Normalization，子层前归一化）**结构，将层归一化放在各分支之前。下面的 **MHA（Multi-Head Attention，多头注意力）**对应前面多组投影并行读取、再拼接投影的计算：

$$u=x+\operatorname{MHA}(\operatorname{LN}_1(x)),\quad
x'=u+\operatorname{FFN}(\operatorname{LN}_2(u))$$

注意：残差通路中的 `x` 没有先被替换成归一化后的数。下一层接收 `x'`，再用新的参数重复交流与加工；输入字数不变，表示不断更新。前面的手算暂时直接用 X，完整模型中的 Block 会先做这里的 LayerNorm。


In [39]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attention = CausalSelfAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model,4*d_model), nn.GELU(), nn.Linear(4*d_model,d_model))

    def forward(self, x, capture=False):
        # Pre-LN：先归一化再进 Attention，残差仍从原 x 直接旁路。
        normalized = self.ln1(x)
        att_update, att_trace = self.attention(normalized, capture=capture)
        after_attention = x + att_update
        # FFN 对每个位置分别加工，但所有位置共享同一组参数。
        ffn_update = self.ffn(self.ln2(after_attention))
        output = after_attention + ffn_update
        trace = dict(input=x,normalized=normalized,attention=att_trace,
                     after_attention=after_attention,ffn_update=ffn_update,output=output) if capture else None
        return output, trace

# 用同一个四维算例检查形状与两次残差，参数仍未训练。
with torch.random.fork_rng():
    torch.manual_seed(19)
    demo_block = DecoderBlock(d_model=4,n_head=2)
with torch.no_grad():
    block_input = torch.tensor(toy_X,dtype=torch.float32)[None,:,:]
    block_out, block_trace = demo_block(block_input,capture=True)
vector_grid(block_trace['attention']['update'][0],list(PREFIX),'Attention 分支给出的更新')
vector_grid(block_trace['after_attention'][0],list(PREFIX),'第一次残差：原输入 + Attention 更新')
vector_grid(block_out[0],list(PREFIX),'第二次残差后：得到完整 Block 输出')
assert torch.allclose(block_trace['after_attention'],block_input+block_trace['attention']['update'])
assert torch.allclose(block_out,block_trace['after_attention']+block_trace['ffn_update'])

# 同一个最后位置的 FFN 算例：4 → 16 → 16 → 4；所有位置共用这些参数。
with torch.no_grad():
    ffn_input = demo_block.ln2(block_trace['after_attention'])[0,-1]
    ffn_expanded = demo_block.ffn[0](ffn_input)
    ffn_activated = demo_block.ffn[1](ffn_expanded)
    ffn_result = demo_block.ffn[2](ffn_activated)
table(['FFN 阶段','完整维度','前 4 项数值'],[
    [label,values.numel(),values[:4].round(decimals=3).tolist()]
    for label,values in [('归一化后的输入',ffn_input),('第一次线性变换',ffn_expanded),
                         ('GELU 非线性',ffn_activated),('第二次线性变换',ffn_result)]])
assert torch.allclose(ffn_result,block_trace['ffn_update'][0,-1],atol=1e-6)


小,-0.502,-0.182,-0.491,-0.443
猫,-0.200,-0.277,-0.421,-0.243
坐,-0.314,0.001,-0.587,-0.355
在,-0.293,-0.125,-0.479,-0.278


小,0.498,-0.182,-0.491,-0.443
猫,-0.100,0.723,-0.421,-0.243
坐,-0.314,0.101,0.413,-0.355
在,0.707,0.875,-0.379,-0.278


小,0.790,-0.723,-0.910,-0.095
猫,-0.125,0.251,-0.482,-0.091
坐,-0.629,0.045,0.348,-0.240
在,0.855,0.262,-0.700,-0.028


归一化后的输入,4,"[0.843999981880188, 1.1410000324249268, -1.0820000171661377, -0.902999997138977]"
第一次线性变换,16,"[-1.1670000553131104, 0.21199999749660492, 0.5789999961853027, -0.550000011920929]"
GELU 非线性,16,"[-0.1420000046491623, 0.12399999797344208, 0.41600000858306885, -0.1599999964237213]"
第二次线性变换,4,"[0.1469999998807907, -0.6129999756813049, -0.32100000977516174, 0.25]"


In [40]:
ln_example = torch.tensor([[1.,2.,3.,4.]])
centered = ln_example-ln_example.mean(dim=-1,keepdim=True)
manual_ln = centered/torch.sqrt(centered.square().mean(dim=-1,keepdim=True)+1e-5)
# 默认初始缩放 gamma=1，偏移 beta=0；训练会改变它们。
actual_ln = nn.LayerNorm(4)(ln_example)
table(['一行输入', '去均值并调整尺度', 'PyTorch 初始 LayerNorm'],
      [[ln_example.tolist()[0],manual_ln.tolist()[0],actual_ln.detach().tolist()[0]]])
assert torch.allclose(manual_ln,actual_ln,atol=1e-6)


"[1.0, 2.0, 3.0, 4.0]","[-1.3416354656219482, -0.4472118318080902, 0.4472118318080902, 1.3416354656219482]","[-1.3416354656219482, -0.4472118318080902, 0.4472118318080902, 1.3416354656219482]"


**先预测**：假设 Attention 和 FFN 两个分支都输出全 0，一个 Pre-LN Block 输出什么？如果没有 Attention，只对带位置的向量反复做逐位置 FFN，最后的「在」能知道前面是猫还是狗吗？

<details><summary>核对与下一步</summary>两次残差都只加 0，Block 输出等于输入。单靠逐位置 FFN 无法把前文传给最后的位置；两句里「在」的初始字符和位置向量相同。现在有了完整 Block，下一步要把处理后的向量接到词表，才能真正预测文字。</details>


## 8 · 2018：GPT：用生成式预训练学习可迁移的表示

**位置**：我们已经有一套能交换和加工上下文的网络。**问题**：每做一个新任务，都必须从头收集大量任务标签吗？**输入 → 输出**：未标注文本的前缀 → 下一 token 分布；之后可适配到其他任务。**下一步**：看见预测、生成与学习的完整计算链。

早期的标签任务很清楚：给一句话一个情感类别，或给每个词一个词性。但新任务就需要相应答案。另一种训练材料却已经藏在文本里：遮去后文，前面的文字就是输入，紧接着的 token 就是答案。下一词预测也被 N-gram 和 RNN 使用过；GPT 的关键并非发明这个目标。

2018 年 6 月发布的 GPT 工作把 **Transformer 与生成式预训练**结合：先在大规模文本上做语言建模，再用较小的监督数据微调到多个任务。可迁移的是学习到的参数与表示。它承接了更早的神经预训练研究，也不是今天聊天产品的全部流程。[官方原始发布与论文，2018-06-11](https://openai.com/index/language-unsupervised/)

$$\mathcal L(\theta)=-\sum_t\log P_\theta(x_t\mid x_{<t}).$$

这里每一项的概率来自同一个网络。与第 2 站的 N-gram 相比，条件前缀与“预测下一 token”的目标很熟悉，估计这个条件分布的方式已经变了：由短窗口计数，变成多层表示和共享参数。**概率模型和深度学习不是彼此排斥的两代技术。**

### 同年的两条预训练路线：生成式与双向编码

2018 年 10 月的 **BERT（Bidirectional Encoder Representations from Transformers，基于 Transformer 的双向编码器表示）**预印本采用双向上下文和遮盖预测等训练任务，是同年另一条预训练路线；它晚于 GPT-1，不是 GPT 必须经过的上一代。GPT 风格用前缀预测后续；BERT 风格可以用遮盖位置两侧的信息。任务设计决定允许看见谁，再影响可用场景。[BERT 原论文](https://arxiv.org/abs/1810.04805)

“预训练”描述训练阶段，“Decoder-only（仅解码器）”描述架构，“自回归”描述概率分解与生成方式，“聊天助手”还涉及后续训练与交互系统。它们不是四个同义词。下面的 MiniGPT 只实现字符级预测与生成，不把六句语料的拟合称为通用预训练的成功。

### 从会续写到按要求回答，还发生了什么？

后续工作的重点也在变化。GPT-2（2019）研究语言模型能否在不做任务专用参数更新时迁移到多个任务；GPT-3（2020）系统研究把少量任务示例放进上下文的使用方式；**InstructGPT（经过指令跟随训练的 GPT，2022）**则用示范与人类偏好等训练信号调整指令跟随行为。这些工作分别改变了迁移方式、输入组织或训练信号，需要区分各自作用的环节。[GPT-2 原报告](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)、[GPT-3 原论文](https://arxiv.org/abs/2005.14165)、[InstructGPT 原论文](https://arxiv.org/abs/2203.02155)

| 环节 | 给模型什么 | 普通流程中会更新参数吗？ |
|---|---|---|
| 语言模型预训练 | 大量文本与自动构造的下一 token 目标 | 会，通过优化损失更新 |
| 监督或偏好相关训练 | 人工示范、比较等额外信号 | 会，依具体训练目标更新 |
| 上下文中的少量示例 | 把示例写在这一次输入里 | 不会；改变输入和中间状态 |
| 自回归生成 | 固定参数下不断增长的前缀 | 不会；每步重新计算分布并选 token |

**先预测**：普通对话中多补一个示例，与执行一次 `optimizer.step()`，哪一个会改变下次全新输入共用的参数？带着这个区别看下面的生成循环和学习循环。


### 8.1 走通生成全景：一次只续出一个 token，为什么能写一整句？

**位置**：把前面分析的 Block 接入完整生成循环。

给定「小猫坐在」，语言模型先为**词表里的每个字符**算一个分数，再把分数变成概率，最后选择一个字符。这个字符接回输入，整个过程再运行一次：

`小猫坐在 → 下一字概率 → 选出「垫」→ 小猫坐在垫 → 再算概率 → ……`

这里的「垫」只是说明循环的例子，尚未训练的模型不一定选它。生成时参数固定，变化的是输入和中间计算结果；训练时我们才改参数。

先认识两类容易混淆的量：**参数**是模型学到并反复使用的规则，比如查表的内容和线性变换的矩阵；**中间表示**是这次输入产生的向量，换输入就可能改变。模型会为每个位置产生输出，续写只取**最后一个位置**的词表分数。第 8.3 节再解释其他位置的输出怎样参与损失计算。


In [41]:
roadmap({0,1,2,3,4})
# 手工分数，仅演示“分数 → 概率”，不是任何模型的预测。
illustration_chars = ['垫', '门', '天', '池', '。']
illustration_logits = torch.tensor([2., 1., 0., -1., -.5])
bars(torch.softmax(illustration_logits, dim=-1), illustration_chars,
     '手工示意：只在这 5 个候选中归一化；真实模型覆盖完整词表')


**先预测**：续写三个字符，是只运行一次模型，还是运行三次？每次需要重新训练吗？

<details><summary>核对与下一步</summary>本模型未启用 **KV cache（Key–Value cache，键值缓存）**，每生成一个字符都重新运行一次前向计算，因此是三次。模型参数不变。下一步把已讲过的输入与 Block 接到输出头。</details>


### 8.2 从隐藏向量到文字：补上真正的生成闭环

**位置**：走出多层 Block，来到输出头。**问题**：每个位置仍只有一个 64 维向量，怎样对应到词表里的每一个字符？

最后再做一次 LayerNorm，然后用线性层把 64 维映射到 `V` 维。`V` 是词表大小，输出的每一项才对应一个候选字符的 **logit（原始分数）**。分数不是概率，可以为负。

对这些词表分数做 Softmax，得到下一个字符的分布。与第 7 站相比，公式相同，选项不同：

| Softmax 用在哪里 | 每个选项是什么 | 接下来做什么 |
|---|---|---|
| Attention 内部 | 当前输入中的一个位置 | 按权重混合该位置的 V 向量 |
| 语言模型输出 | 整个词表中的一个字符 | 选择一个字符，接回输入 |

因此可以把这一段代码读成：`(B,T,D) → (B,T,V)`；训练保留所有 T 个位置，生成只取最后一个位置。模型为每个位置都计算一个词表分布。**位置 t 的分布预测 t+1**；生成新字符时只取最后一个位置，前面的分布留给训练使用。


In [42]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_head=4, n_layer=2, block_size=32):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size,d_model)
        self.pos_emb = nn.Embedding(block_size,d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model,n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model,vocab_size,bias=False)

    def forward(self, ids, capture=False):
        if ids.ndim != 2 or not 1 <= ids.shape[1] <= self.block_size:
            raise ValueError(f'输入需要是 (B,T)，且 1 ≤ T ≤ {self.block_size}。')
        # ids 是 (B,T)；查表后 token/position 都是 (B,T,D)。
        token_vectors = self.tok_emb(ids)
        position_vectors = self.pos_emb(torch.arange(ids.shape[1],device=ids.device))[None,:,:]
        x = token_vectors + position_vectors
        trace = dict(token_vectors=token_vectors,position_vectors=position_vectors,input=x,blocks=[]) if capture else None
        for block in self.blocks:
            # 每层都保持 (B,T,D)，只是表示内容不断更新。
            x, block_trace = block(x,capture=capture)
            if capture:
                trace['blocks'].append(block_trace)
        normalized = self.ln_f(x)
        # 输出头把每个位置的 D 维表示映射成 V 个候选字符分数。
        logits = self.head(normalized)
        if capture:
            trace.update(normalized=normalized,logits=logits)
            return logits, trace
        return logits

    @torch.no_grad()
    def inspect(self, ids):
        # 和训练/生成共用同一条 forward；并非另外编造一组可视化结果。
        return self(ids,capture=True)[1]

torch.manual_seed(7)
model = MiniGPT(VOCAB_SIZE).cpu()
training_status = '尚未训练'
with torch.no_grad():
    initial_logits = model(prefix_ids)
    initial_probs = F.softmax(initial_logits[0,-1],dim=-1)
initial_state = copy.deepcopy(model.state_dict())
roadmap({1,2,3})
print('输入 ID 形状：',tuple(prefix_ids.shape),'=（B：句子数，T：位置数）')
print('logits 形状：',tuple(initial_logits.shape),'=（B，T，V：词表大小）')
bars(initial_probs,CHARS,'未训练模型：最后位置的下一字符概率',top=8)


输入 ID 形状： (1, 4) =（B：句子数，T：位置数）
logits 形状： (1, 4, 23) =（B，T，V：词表大小）


#### 在实际模型中追踪同一位置

接下来全是这个 MiniGPT **实际算出的值**，与前面的四维手工算例分开观察。选择层、头和位置，顺着输入 → 匹配 → 混合 → 更新 → 词表概率看过去。为了便于阅读，向量表只显示前几列，计算使用完整维度。

“猫前缀”与“狗前缀”最后都为「在」，位置也相同；切换前缀时，字符与位置表给它的初始向量相同，经过 Attention 后的表示则可能不同。未训练时这些差异不代表已经学到了有用关系。实验室训练完成后，这个面板会刷新为训练后的模型，并显示状态。

**先预测**：选择位置 1「猫」，它的词表概率在预测「猫」本身，还是它后面的字符？它能读取位置 2「坐」吗？


In [43]:
def inspect_real(prefix, layer, head, position):
    ids = encode(prefix)
    trace = model.inspect(ids)
    block = trace['blocks'][layer]
    att = block['attention']
    display(HTML(f'<b>MiniGPT 实际计算：{html.escape(training_status)}</b>；第 {layer+1} 层，第 {head+1} 头，位置 {position}「{html.escape(prefix[position])}」'))
    vector_grid(trace['token_vectors'][0,position],[prefix[position]],'查字符表（同 ID 总是同一行）')
    vector_grid(trace['position_vectors'][0,position],[f'位置 {position}'],'查位置表')
    vector_grid(block['input'][0],list(prefix),'这一层输入：第一层为两表之和，后续层为前一层输出')
    vector_grid(att['q'][0,head,position],[prefix[position]],'当前头的 Q；输入先经过 LayerNorm，再做投影')
    table(['来源位置','缩放后的匹配分数','Mask 后','注意力权重'],[
        [f'{j}：{ch}',float(att['scores'][0,head,position,j]),
         '不可读' if j>position else float(att['masked_scores'][0,head,position,j]),
         float(att['weights'][0,head,position,j])] for j,ch in enumerate(prefix)])
    vector_grid(att['k'][0,head],list(prefix),'当前头的 K：与 Q 匹配')
    vector_grid(att['v'][0,head],list(prefix),'当前头的 V：被权重混合的内容')
    vector_grid(att['mixed'][0,head,position],[prefix[position]],'当前头取回的内容：完整 16 维，只展示前 8 维')
    vector_grid(torch.stack([block['input'][0,position],att['update'][0,position],
                            block['after_attention'][0,position],block['ffn_update'][0,position],
                            block['output'][0,position]]),
                ['原输入','多头拼接并投影后的更新','第一次残差后','FFN 更新','完整 Block 输出'],
                '看同一位置怎样逐步更新')
    probs = F.softmax(trace['logits'][0,position],dim=-1)
    bars(probs,CHARS,f'经过全部层后的词表概率：位置 {position} 的输出预测下一位置',top=8)
    print('上面的词表概率始终来自完整模型；换“层/头”只改变中间观察窗口。')

In [44]:
real_ui = panel('real_model',inspect_real,OrderedDict(
    prefix=widgets.Dropdown(options=['小猫坐在','小狗坐在','小鸟飞在'],value=PREFIX,description='前缀'),
    layer=widgets.Dropdown(options=[('第 1 层',0),('第 2 层',1)],value=0,description='观察层'),
    head=widgets.Dropdown(options=[(f'第 {h+1} 头',h) for h in range(4)],value=0,description='观察头'),
    position=widgets.Dropdown(options=[(f'位置 {i}',i) for i in range(4)],value=3,description='观察位置')))

#### 从分布选一个字符，再把它接回输入

**采样**按概率随机抽取候选；**贪心选择**直接取概率最大的一项。两者都不更新模型参数。

生成时可把词表 logits 除以温度 `T` 后再做 Softmax。较低温度让分布更集中，但只要仍采用采样，就不是“保证取最大值”。较高温度让分布更平。这里调的是**输出词表分布**的温度，不修改 Block 内的注意力公式。

下面每一步都列出本轮真正使用的前缀、概率和选中的字符。超过 32 个字符时，本实验模型只取最近 32 个，并将该窗口的位置从 0 重新编号；这是一种简单滑窗实现，不代表所有 GPT 的长上下文处理方式。


In [45]:
@torch.no_grad()
def generate_steps(net, prompt, count=3, temperature=1., greedy=False, seed=11):
    if not math.isfinite(temperature) or temperature <= 0:
        raise ValueError('温度必须是正的有限数。')
    if not 1 <= count <= 12:
        raise ValueError('本实验每次生成 1 到 12 个字符。')
    ids = encode(prompt)
    rng = torch.Generator().manual_seed(seed)
    records = []
    for step in range(count):
        # 本轮只把已有前缀送进模型；超长时保留最后 block_size 个词元。
        context = ids[:,-net.block_size:]
        # 模型为所有位置给出分数，生成时只取最后一个位置。
        logits = net(context)[0,-1]
        # 温度只改变输出分布，不更新参数；multinomial 按概率抽样。
        probs = F.softmax(logits/temperature,dim=-1)
        next_id = int(probs.argmax()) if greedy else int(torch.multinomial(probs,1,generator=rng).item())
        best = torch.topk(probs,min(3,VOCAB_SIZE))
        records.append(dict(context=decode(context[0]),token=ITOS[next_id],
                            probability=float(probs[next_id]),
                            top3='；'.join(f'{ITOS[int(i)]} {float(p):.3f}' for p,i in zip(best.values,best.indices))))
        ids = torch.cat([ids,torch.tensor([[next_id]])],dim=1)
    return decode(ids[0]),records

before_text,before_records = generate_steps(model,PREFIX)
table(['轮次','本轮输入窗口','前三个候选','本次选择','其概率'],
      [[i+1,r['context'],r['top3'],r['token'],r['probability']] for i,r in enumerate(before_records)],
      '未训练模型的实际生成轨迹')
print('接回后的文本：',before_text)


1,小猫坐在,飞 0.092；天 0.088；鸟 0.069,天,0.088
2,小猫坐在天,趴 0.184；飞 0.122；游 0.077,子,0.035
3,小猫坐在天子,小 0.154；。 0.150；垫 0.053,小,0.154


接回后的文本： 小猫坐在天子小


In [46]:
def render_generation(prompt, temperature, greedy, count):
    text,records = generate_steps(model,prompt,count=count,temperature=temperature,greedy=greedy)
    display(HTML(f'<b>实际模型：{html.escape(training_status)}</b>；采样使用固定随机种子，方便比较温度。'))
    table(['轮次','输入窗口','前三候选','选择','概率'],[
        [i+1,r['context'],r['top3'],r['token'],r['probability']] for i,r in enumerate(records)])
    print('生成结果：',text)

In [47]:
generation_ui = panel('generation',render_generation,OrderedDict(
    prompt=widgets.Text(value=PREFIX,description='前缀'),
    temperature=widgets.FloatSlider(value=1.,min=.2,max=2.,step=.1,description='输出温度',continuous_update=False),
    greedy=widgets.Checkbox(value=False,description='贪心选择（不随机抽样）'),
    count=widgets.IntSlider(value=3,min=1,max=12,description='生成字数',continuous_update=False)))

<details><summary>核对与下一步</summary>位置 1「猫」的输出预测位置 2 的字符，但不能读取位置 2 的输入。在本例里正确标签为「坐」，模型当下的概率未必把它排在第一。最后位置的输出没有已知后续字符可读，只能根据前文预测。现在能续字了，下一步要问：这些参数怎样从随机数变得有用？</details>


### 8.3 学习：原文自己提供下一字的答案

**位置**：从“用固定参数预测”转到“用答案调整参数”。**问题**：我们怎样告诉模型哪些预测更好？

取已知文本「小猫坐在垫子上。」。输入去掉最后一个字，标签去掉第一个字，就得到一对错开一位的序列：

- 输入：`小 猫 坐 在 垫 子 上`
- 标签：`猫 坐 在 垫 子 上 。`

位置 0 用「小」预测「猫」；位置 3 用「小猫坐在」预测「垫」。一次前向能为所有位置产生分数，因此**训练可并行计算多个下一字任务**。因果 Mask 保证位置 t 不能读取 t+1 的答案；它不是把正确标签从词表里删掉。

生成时未来文本还没产生，所以要逐字符循环。训练时已知后续文本，可以把它放在张量里，但只能让各位置读取自己的前缀。


In [48]:
lesson_ids = encode(SENTENCES[0])
# 右移一位：输入“小 猫 坐”，标签“猫 坐 在”。
# 所以位置 t 的 logits 对应标签 y[t]，任务就是预测下一个词元。
lesson_x,lesson_y = lesson_ids[:,:-1],lesson_ids[:,1:]
table(['位置 t','本位置输入','允许读取的前缀','该输出的正确下一字'],[
    [t,ITOS[int(lesson_x[0,t])],decode(lesson_x[0,:t+1]),ITOS[int(lesson_y[0,t])]]
    for t in range(lesson_x.shape[1])])
assert torch.equal(lesson_x[:,1:],lesson_y[:,:-1])


0,小,小,猫
1,猫,小猫,坐
2,坐,小猫坐,在
3,在,小猫坐在,垫
4,垫,小猫坐在垫,子
5,子,小猫坐在垫子,上
6,上,小猫坐在垫子上,。


#### 怎样给一轮预测打分：交叉熵

如果正确字符是「垫」，模型给它的概率为 `p`，这一位置的**损失（loss）**就是 `−log(p)`。给正确答案的概率越高，损失越低：`p=0.5` 时约 0.693，`p=0.1` 时约 2.303。损失不只检查最终抽到了什么字符，而是检查整个预测分布给正确答案留了多少概率。

把所有训练位置的损失取平均，得到这一批次的交叉熵；例如正确概率为 `.5` 时，单个位置的损失是 $-\log(.5)\approx.693$。PyTorch 的 `cross_entropy` **直接接收 logits**，内部用稳定的 log-softmax 计算；使用时应传入原始分数，无需先做 Softmax。


In [49]:
with torch.no_grad():
    lesson_logits = model(lesson_x)
    # logits -> log 概率；gather 只取每个位置“正确下一字”的那一列。
    log_probs = F.log_softmax(lesson_logits,dim=-1)
    target_log_probs = log_probs.gather(-1,lesson_y.unsqueeze(-1)).squeeze(-1)
    # 交叉熵就是正确标签的 -log(p)，再对所有位置求平均。
    per_position_loss = -target_log_probs
    manual_loss = per_position_loss.mean()
    # PyTorch 版本直接接收 logits，内部完成稳定的 log-softmax。
    builtin_loss = F.cross_entropy(lesson_logits.reshape(-1,VOCAB_SIZE),lesson_y.reshape(-1))
table(['位置','正确下一字','模型分配给它的概率','该位置 −log(p)'],[
    [t,ITOS[int(lesson_y[0,t])],float(target_log_probs[0,t].exp()),float(per_position_loss[0,t])]
    for t in range(lesson_x.shape[1])])
print('手算平均损失：',float(manual_loss),'；cross_entropy：',float(builtin_loss))
assert torch.allclose(manual_loss,builtin_loss,atol=1e-6)


0,猫,0.011,4.477
1,坐,0.070,2.664
2,在,0.096,2.338
3,垫,0.055,2.903
4,子,0.038,3.271
5,上,0.020,3.927
6,。,0.056,2.888


手算平均损失： 3.2098288536071777 ；cross_entropy： 3.2098286151885986


#### 错误怎样改动参数：一次前向、反向、更新

梯度给出损失对每个参数的局部变化率。最基本的梯度下降用 `参数 ← 参数 − 学习率 × 梯度`，往局部降低损失的方向走。学习率决定每次改多少；它与生成温度是两回事。

本 Notebook 每轮代码的顺序是 `forward → loss → zero_grad → backward → step`；`zero_grad` 放在前向之后也可以，因为它只清理旧梯度。前四步只计算，`step` 才改变参数。反向传播把输出损失沿着同一条计算链传回输出头、FFN、Attention 和两张输入表。训练根据预测损失，调整产生匹配关系与传递内容的参数。这里使用 **AdamW（Adam with decoupled weight decay，采用解耦权重衰减的 Adam 优化器）**；其中 **Adam** 源于 **Adaptive Moment Estimation（自适应矩估计）**。它利用梯度历史调整更新步长，并单独施加权重衰减，具体更新量与最简单的梯度下降公式有所不同。[算法原论文](https://arxiv.org/abs/1711.05101)

下面复制一份模型，只做一次更新，观察真实损失和参数变化。副本用于这个实验，末尾实验室中的完整训练从原模型的初始参数开始。一次更新不保证所有位置都变好，也不保证从此总能生成目标句子。


In [50]:
step_model = copy.deepcopy(model)
step_optimizer = torch.optim.AdamW(step_model.parameters(),lr=1e-3)
tracked_names = ['tok_emb.weight','pos_emb.weight','blocks.0.attention.qkv.weight','blocks.0.ffn.0.weight','head.weight']
tracked_before = {name:p.detach().clone() for name,p in step_model.named_parameters() if name in tracked_names}

step_logits = step_model(lesson_x)                       # 前向：每个位置预测下一字。
step_loss = F.cross_entropy(step_logits.reshape(-1,VOCAB_SIZE),lesson_y.reshape(-1))
step_optimizer.zero_grad(set_to_none=True)               # 清掉上轮梯度。
step_loss.backward()                                    # 反向：沿计算图求参数梯度。
grad_norms = {name:float(p.grad.norm()) for name,p in step_model.named_parameters() if name in tracked_names}
step_optimizer.step()                                   # 优化器按梯度真正改动参数。

with torch.no_grad():
    loss_after_one_step = F.cross_entropy(step_model(lesson_x).reshape(-1,VOCAB_SIZE),lesson_y.reshape(-1))
table(['参数','这次梯度的范数','更新前后差值的范数'],[
    [name,grad_norms[name],float((p-tracked_before[name]).norm().detach())]
    for name,p in step_model.named_parameters() if name in tracked_names])
print(f'这次运行：更新前损失={float(step_loss.detach()):.4f}，更新后损失={float(loss_after_one_step):.4f}')


tok_emb.weight,0.156,0.021
pos_emb.weight,0.156,0.021
blocks.0.attention.qkv.weight,0.398,0.111
blocks.0.ffn.0.weight,0.428,0.128
head.weight,2.952,0.038


这次运行：更新前损失=3.2098，更新后损失=2.8078


**先预测**：把 `optimizer.step()` 去掉，损失仍能算出来吗？参数会变吗？如果把右移标签改成输入本身，模型学到的任务还是“预测下一字”吗？

<details><summary>核对与下一步</summary>仍能前向算损失、反向算梯度，但不执行更新就不会改参数。标签等于输入会变成在当前位置复述输入，与下一字任务不同。现在我们看清一次学习的全过程；后面的实验室把这个过程重复多次。</details>


## 9 · 回望：变化的是表示、依赖与学习方式

**位置**：已经能从历史问题走到 GPT 的具体计算。**问题**：这些方法分别改变了什么？**输入 → 输出**：任务、假设与证据 → 有根据的模型选择。**接下来**：选做完整训练实验，再用原始材料扩展阅读。

| 路线 | 从什么学或写入知识 | 输出是什么 | 一个核心限制 |
|---|---|---|---|
| 规则与显式表示 | 人编写知识、词典、规则 | 结构、动作或任务结果 | 覆盖范围与维护成本 |
| 人工标注 | 人定义任务并提供参考答案 | 训练数据，不是一种预测架构 | 成本、歧义与标注范围 |
| N-gram | 语料中的短串计数 | 下一 token 分布 | 窗口与稀疏性 |
| 本例 HMM | 标签转移、词发射概率 | 给定词序列的标签路径 | 条件独立假设与状态设计 |
| 神经窗口模型 | 联合学习向量和预测函数 | 下一 token 分布 | 固定窗口与数据需求 |
| RNN / LSTM | 共享循环参数与状态更新 | 状态、标签或下一 token 分布 | 长程优化、压缩与顺序依赖 |
| Transformer | 多层 attention、FFN 等参数 | 上下文表示与任务输出 | 标准全注意力开销、数据与泛化 |
| GPT 式预训练 | 文本提供的下一 token 目标 | 学习参数、条件分布 | 训练目标不保证事实性或理解 |

语义角色标注可以由神经网络完成，规则可以与统计方法组合，RNN 也可以预训练并做语言建模。不同方法可以长期并存，也可以组合使用。比较它们时，可以依次问：**任务要什么输出？模型允许依赖什么？知识来自哪里？什么证据说明学会了？**

### 把“能看见谁”画出来

下面对照两种自注意力读取范围。蓝格表示一个位置在这种子层中允许读取的输入位置，不是训练出的注意力大小。BERT 的遮盖位置没有被抹掉，只是输入换成了占位 token；可见两侧并不意味着读取到了被遮住的原 token。Encoder–Decoder 还需另一张跨源句/目标句的交叉注意力表，这里仅对照自注意力子层。

**先预测**：如果把 GPT 的因果 Mask 改成双向，再用相同右移标签训练，会不会让本位置直接读到下一字答案？


In [51]:
def visibility_mask(kind,length):
    return np.tril(np.ones((length,length),dtype=bool)) if kind=='GPT 因果' else np.ones((length,length),dtype=bool)

In [52]:
def render_visibility(kind):
    mask = visibility_mask(kind,4)
    parts = [svg_text(290,25,kind+'：行查询、列来源',size=16)]
    for i,ch in enumerate(PREFIX):
        parts.append(svg_text(135,98+i*52,f'{i} {ch}',size=14,anchor='end'))
        parts.append(svg_text(185+i*52,60,f'{i} {ch}',size=14))
        for j in range(4):
            color = BLUE if mask[i,j] else '#e7ecef'
            parts.append(f'<rect x="{165+j*52}" y="{75+i*52}" width="40" height="40" fill="{color}"/>')
    show_svg(parts,520,310,'GPT 与双向 Encoder 的允许读取关系，不是注意力权重热图')
    print('GPT：本位置不能看下一位置。双向 Encoder：用于与其匹配的训练任务，不能直接照搬下一字标签。')

In [53]:
visibility_ui = panel('visibility',render_visibility,OrderedDict(
    kind=widgets.ToggleButtons(options=['GPT 因果','BERT / Encoder 双向'],value='GPT 因果')))

### 讨论：把历史问题与计算机制联系起来

1. 规则输出了一个错误施事时，改变的是文字、人工答案，还是建模假设？
2. N-gram 为什么既怕窗口太短，也怕窗口太长？平滑解决了哪部分问题？
3. HMM 为什么不能只逐个选最大标签概率？它输出的标签概率与 GPT 的词表概率能直接比较吗？
4. 词向量共享了什么？RNN 的状态与模型参数有什么不同？
5. 2014 年的 Attention 和 2017 年 Transformer 的关系是什么？Attention 的输出仍然是什么？
6. FFN 在哪里加工信息？只有 Attention 为什么还不足以说明完整 Block？
7. GPT 继承了哪些旧目标，又改变了哪些表示与学习方式？预训练和提示输入是否都会改变参数？

<details><summary>核对关键关系</summary>
人工参考答案与模型预测要分开；短窗口丢依赖，长窗口少样本，简单平滑只处理零计数；HMM 要在路径联合概率中考虑相邻标签，不同任务的结果空间不同；Embedding 与预测函数共享参数，RNN 状态随当前文本变化、参数由训练更新；Attention 先用于 RNN 翻译，再成为 Transformer 主干的重要部分，输出仍是向量；FFN 在各位置独立加工已汇集的表示；GPT 继承条件概率与下一词目标，结合 Transformer 和可迁移训练。普通提示改变输入和激活，不执行参数更新。
</details>


## 实验室 A · 组装与训练 MiniGPT：把同一条链重复运行

**位置**：从单条样本走向多个训练窗口。**问题**：怎样得到一批包含不同上下文的下一字任务，并观察学习前后的实际变化？

我们已经逐节定义了 Attention、Block、输出头及完整模型；这里使用同一个 MiniGPT 完成训练与可视化。训练批次从练习语料中取长度 32 的窗口，标签整体右移一位。`B=16` 表示一批有 16 个窗口，`T=32` 表示每个窗口 32 个位置，`D=64` 是每个位置的表示宽度。

全模型有 2 个 Block、每层 4 个头，使用可学习位置表；未使用 **Dropout（随机失活）**，以便观察和核对数值。我们训练 180 步，显示批次损失，再用同一组已知练习句子比较训练前后。**这组句子也在训练语料中，比较只说明训练集拟合，不是验证集成绩。**

本单元每次运行都会重置为前面保存的初始参数，并重置抽样随机数，方便重复实验。控件面板随后刷新为训练后的实际模型。运行时间取决于 CPU。


In [54]:
model.load_state_dict(initial_state)
model.train()
training_status = '训练中'
training_data = encode(CORPUS)[0]
batch_rng = torch.Generator().manual_seed(23)

def get_batch(batch_size=16,block_size=32):
    # 窗口要多取一个字符，才能给最后一个输入位置配标签。
    starts = torch.randint(len(training_data)-block_size,(batch_size,),generator=batch_rng)
    x = torch.stack([training_data[i:i+block_size] for i in starts])
    y = torch.stack([training_data[i+1:i+block_size+1] for i in starts])
    return x,y

@torch.no_grad()
def practice_loss(net):
    total, count = 0.,0
    for text in SENTENCES:
        ids = encode(text)
        x,y = ids[:,:-1],ids[:,1:]
        total += float(F.cross_entropy(net(x).reshape(-1,VOCAB_SIZE),y.reshape(-1),reduction='sum'))
        count += y.numel()
    return total/count

before_practice_loss = practice_loss(model)
before_text,before_records = generate_steps(model,PREFIX)  # 重跑此单元时，训练前续写也从初始参数重新计算。
with torch.no_grad():
    before_training_probs = F.softmax(model(prefix_ids)[0,-1],dim=-1).clone()
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-3)
loss_history = []
progress = display(HTML('开始训练……'),display_id=True)
for step in range(1,181):
    xb,yb = get_batch()                              # 取一批错开一位的训练样本。
    logits = model(xb)                               # 同时得到所有位置的下一字分数。
    loss = F.cross_entropy(logits.reshape(-1,VOCAB_SIZE),yb.reshape(-1))
    optimizer.zero_grad(set_to_none=True)            # 清掉上一批的梯度。
    loss.backward()                                  # 误差从输出头传回所有 Block。
    optimizer.step()                                 # 用 AdamW 更新参数。
    loss_history.append(float(loss.detach()))
    if step == 1 or step % 30 == 0:
        progress.update(HTML(f'<b>训练步 {step}/180；当前批次损失={loss_history[-1]:.4f}</b>'+loss_plot(loss_history)))

model.eval()
training_status = '已完成 180 步训练'
after_practice_loss = practice_loss(model)
with torch.no_grad():
    after_training_probs = F.softmax(model(prefix_ids)[0,-1],dim=-1).clone()
after_text,after_records = generate_steps(model,PREFIX)
table(['对照项','训练前','训练后'],[
    ['已知练习句子的逐字符平均交叉熵',before_practice_loss,after_practice_loss],
    ['同前缀、同随机种子的 3 字续写',before_text,after_text]])
for key in ['real_model','generation']:
    PANELS[key]['refresh']()


已知练习句子的逐字符平均交叉熵,3.345,0.309
同前缀、同随机种子的 3 字续写,小猫坐在天子小,小猫坐在垫子上


In [55]:
# 同一个前缀、同一完整词表：对比的是实际模型分布。
bars(before_training_probs,CHARS,'训练前：小猫坐在 → 下一字符',top=8)
bars(after_training_probs,CHARS,'训练后：小猫坐在 → 下一字符',top=8)
table(['轮次','本轮输入','前三候选','选择','概率'],[
    [i+1,r['context'],r['top3'],r['token'],r['probability']] for i,r in enumerate(after_records)],'训练后生成闭环')
print('再回第 8 站，切换猫/狗前缀、层和头；面板当前使用训练后的参数。')


1,小猫坐在,垫 0.994；门 0.003；躺 0.001,垫,0.994
2,小猫坐在垫,子 0.998；狗 0.000；垫 0.000,子,0.998
3,小猫坐在垫子,上 0.998；中 0.000；里 0.000,上,0.998


再回第 8 站，切换猫/狗前缀、层和头；面板当前使用训练后的参数。


### A.1 核对模型计算与公式是否一致

除了观察训练损失，还需要核对完整前向形状、每行权重的和、未来位置的零权重、V 的混合、两次残差、输出投影，以及改动未来字符对更早输出的影响。它们检查计算关系，不要求某个头一定看「猫」，也不要求随机采样固定续出「垫子上」。


In [56]:
trace = model.inspect(prefix_ids)
T = prefix_ids.shape[1]
assert trace['logits'].shape == (1,T,VOCAB_SIZE)
assert torch.allclose(trace['input'],trace['token_vectors']+trace['position_vectors'])
for block in trace['blocks']:
    att = block['attention']
    assert torch.allclose(att['weights'].sum(-1),torch.ones(1,4,T),atol=1e-6)
    future = torch.ones(T,T,dtype=torch.bool).triu(1)
    assert torch.count_nonzero(att['weights'][...,future]) == 0
    assert torch.allclose(att['mixed'],att['weights']@att['v'],atol=1e-6)
    assert torch.allclose(block['after_attention'],block['input']+att['update'],atol=1e-6)
    assert torch.allclose(block['output'],block['after_attention']+block['ffn_update'],atol=1e-6)
assert torch.allclose(trace['logits'],model.head(trace['normalized']),atol=1e-6)

# 修改位置 2、3 的输入，位置 0、1 的输出应保持不变。
changed = prefix_ids.clone()
changed[0,2:] = torch.tensor([STOI['狗'],STOI['门']])
with torch.no_grad():
    assert torch.allclose(model(prefix_ids)[:,:2],model(changed)[:,:2],atol=1e-6)
xb_check,yb_check = get_batch(batch_size=2)
assert torch.equal(xb_check[:,1:],yb_check[:,:-1])
assert all(math.isfinite(x) for x in loss_history)
assert math.isfinite(after_practice_loss)
print('核对通过：形状、归一化、因果隔离、内容混合、残差、输出映射和标签右移。')
print(f'已知练习句子的实际损失：{before_practice_loss:.4f} → {after_practice_loss:.4f}')


核对通过：形状、归一化、因果隔离、内容混合、残差、输出映射和标签右移。
已知练习句子的实际损失：3.3446 → 0.3086


**先预测并解释**：如果训练损失降得很低，能不能说明它已经学会自然语言？如果某一注意力头的热图不符合你的直觉，能不能直接判断这个模型训练失败？

<details><summary>核对</summary>这几句话高度重复，低训练损失可能主要来自记忆。需要独立数据才能评价泛化。模型还有其他头、其他层、FFN 和残差；单张热图不足以判断整体是否学好。热图可用于追踪计算，但每个头的功能还需要进一步的实验分析。</details>


### A.2 核对历史方法算例的计算关系

模型的训练表现与公式是否按预期实现，是两种检查。下面核对计数、HMM、连续表示、循环递推、自动微分和可见区的计算关系。这些断言不能证明模型理解语言，用于检查实现是否符合前文公式。


In [57]:
_,cat_counts,cat_probs = ngram_distribution('小猫坐在',2,0.)
_,dog_counts,dog_probs = ngram_distribution('小狗坐在',2,0.)
assert np.array_equal(cat_counts,dog_counts)
assert np.allclose(cat_probs,dog_probs)
_,_,unseen = ngram_distribution('小猫趴在',4,0.)
assert unseen is None
_,_,smoothed = ngram_distribution('小猫趴在',4,.1)
assert np.allclose(smoothed,np.ones(VOCAB_SIZE)/VOCAB_SIZE)
for near in (0., .5, 1.):
    _, _, _, h, logits, probs = embedding_distribution('猫', '趴', near)
    assert np.allclose(logits, window_U@h+window_a) and np.isclose(probs.sum(), 1.)
_, _, _, _, _, close = embedding_distribution('猫', '趴', 1.)
_, _, _, _, _, seen = embedding_distribution('猫', '坐', 1.)
_, _, _, _, _, apart = embedding_distribution('猫', '趴', 0.)
assert window_candidates[int(close.argmax())] == '在'
assert window_candidates[int(seen.argmax())] == '在'
assert close[0] > apart[0]+.5
_, _, _, _, _, sit = embedding_distribution('坐', '在', 1.)
assert np.isclose(sit[1], sit[2]) and sit[1] > sit[0]
_, _, _, _, _, sky = embedding_distribution('飞', '在', 1.)
assert window_candidates[int(sky.argmax())] == '天'
assert not np.allclose(rnn_states('小猫坐在')[-1],rnn_states('小狗坐在')[-1])
sens,retained = memory_curves(.7,.95,12)
scalar=.2
manual=1.
for t in range(12):
    scalar=math.tanh(.7*scalar)
    manual *= .7*(1-scalar**2)
assert np.isclose(sens[-1],manual,rtol=1e-5)
assert np.isclose(retained[-1],.95**12)
for angle in [0,90,180,270]:
    q,scores,a,c = cross_attention(angle)
    assert np.isclose(a.sum(),1.) and np.allclose(c,a@cross_values)
assert not visibility_mask('GPT 因果',4)[0,1]
assert visibility_mask('BERT / Encoder 双向',4)[0,1]
assert all(state['error'] is None for state in PANELS.values())
print('核对通过：计数/平滑、连续投影、循环状态、梯度链、加权读取与架构可见区。HMM 在本节内部另有路径核对。')


核对通过：计数/平滑、连续投影、循环状态、梯度链、加权读取与架构可见区。HMM 在本节内部另有路径核对。


## 继续阅读：按问题回到原始材料

| 问题 | 原始材料 | 阅读时留意 |
|---|---|---|
| 语言的统计结构 | [Shannon，1948](https://people.math.harvard.edu/~ctm/home/text/others/shannon/entropy/entropy.pdf)；[Weaver，1949](https://www.mt-archive.net/50/Weaver-1949.pdf) | 统计序列与机器翻译设想，不等于当代大模型 |
| 语言接到显式世界 | [Winograd，1971](https://dspace.mit.edu/entities/publication/3406b826-5a41-4b72-9430-7876c0d5405e) | 限定的积木世界如何支持推理与指代 |
| 概率状态序列 | [Rabiner，1989](https://www.cs.cornell.edu/courses/cs481/2004fa/rabiner.pdf) | 语音应用；区分序列概率、解码与参数学习 |
| 标注是什么 | [Penn Treebank，1993](https://aclanthology.org/J93-2004/)；[PropBank，2005](https://aclanthology.org/J05-1004/) | 词性、句法和语义角色是不同层次 |
| 从状态到记忆门 | [Elman，1990](https://doi.org/10.1207/S15516709COG1402_1)；[LSTM，1997](https://www.bioinf.jku.at/publications/older/2604.pdf) | RNN 的历史早于 2010 年代 |
| 共享连续表示 | [Bengio 等，2003](https://jmlr.org/papers/v3/bengio03a.html) | 联合学习表示与概率函数，仍使用固定窗口 |
| 按需读源句 | [Bahdanau 等，2014 / 2015](https://arxiv.org/abs/1409.0473) | 原始加性 attention 与后来的点积 attention 不同 |
| Transformer | [Vaswani 等，2017](https://arxiv.org/abs/1706.03762) | 原版 Encoder–Decoder、并行化与位置对开销 |
| 生成式预训练 | [GPT，2018 年 6 月](https://openai.com/index/language-unsupervised/) | 大规模语言建模之后再做监督微调 |
| 双向预训练分支 | [BERT，2018 年 10 月](https://arxiv.org/abs/1810.04805) | 训练目标和允许读取的上下文 |

### 选读：从简化实验走向实际系统

- **Token**：真实模型常用词、子词或字节片段。改变分词会改变序列长度与词表，ID → 向量 → 上下文表示 → 词表分数的关系仍在。
- **位置**：原版 Transformer 使用正弦/余弦位置编码，本模型使用可学习位置表；相对位置或 **RoPE（Rotary Position Embedding，旋转位置嵌入）**等方法有不同的计算方式，需要分别理解其如何把位置信息引入计算。[RoPE 原论文](https://arxiv.org/abs/2104.09864)。
- **生成效率**：缓存 K/V 可以减少重复计算。自回归生成仍依赖已经选出的 token；训练所有位置并行不等于一次生成全部文本。
- **训练阶段**：预训练、监督微调、偏好训练各有目标。小语料拟合只展示优化过程，不证明可靠知识、泛化或对话能力。
- **实现核对**：[PyTorch cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html) 接收 logits；[LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html) 沿特征维计算，并有可学习缩放与偏移。

模型参数仅保存在当前内核内存中；关闭内核后，重新运行可恢复实验。参考资料用于扩展阅读，计算过程无需访问这些链接。
